# ARCHS4 Biology Validation

Identify tissue- and cell-type-enriched LVs in ARCHS4 bulk RNA-seq data.
For each query, samples are matched by keyword in `source_name_ch1` / `characteristics_ch1`,
LVs are ranked by enrichment ratio in the top 1%, and significant hits are linked to pathways and GWAS traits.

💡 **Environment:** `clamp-analyses`

In [1]:
import re
import textwrap

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

from pyprojroot.here import here

import rpy2.robjects as ro
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri

readRDS = ro.r["readRDS"]

## Settings

In [2]:
# ── pathway display thresholds ────────────────────────────────────────────────
FDR_THRESH_PATHWAYS  = 0.1    # pathway FDR cutoff
AUC_THRESH_PATHWAYS  = 0.6    # pathway AUC cutoff
SC_PROB_THRESH       = 0.5    # exclude samples with single-cell probability ≥ this

# ── LV selection (data-driven) ─────────────────────────────────────────────────
TOP_PERC             = 0.01   # top-1% window defines the "enriched" zone
FDR_THRESH_LVS       = 0.05   # Mann-Whitney FDR threshold for LV selection
ENRICH_RATIO_THRESH  = 2.0    # minimum enrichment ratio (observed / expected)
MIN_SAMPLES          = 10     # skip queries with fewer samples than this
MIN_N_IN_TOP1PCT     = 20     # minimum samples in top 1% for an LV to be selected

## Load archs4 model

In [3]:
def extract_B_matrix(rds_obj):
    B_matrix = rds_obj.rx2("B")
    with localconverter(ro.default_converter + pandas2ri.converter):
        B_values = ro.conversion.rpy2py(B_matrix)
    return pd.DataFrame(
        data=B_values,
        index=B_matrix.rownames if B_matrix.rownames else None,
        columns=B_matrix.colnames if B_matrix.colnames else None,
    )


def extract_summary_matrix(rds_obj):
    summary_matrix = rds_obj.rx2("summary")
    with localconverter(ro.default_converter + pandas2ri.converter):
        summary_values = ro.conversion.rpy2py(summary_matrix)
    return pd.DataFrame(
        data=summary_values,
        index=summary_matrix.rownames if summary_matrix.rownames else None,
        columns=summary_matrix.colnames if summary_matrix.colnames else None,
    )


model_rds = readRDS(str(here("output/archs4/c2cp_coverage_rs100_seed_1/CLAMPfull_C2CP.rds")))

# B matrix: rows = LVs, cols = samples → transpose to samples × LVs
B_df      = extract_B_matrix(model_rds)
lv_matrix = B_df.T

# Summary matrix: pathway–LV associations
summary_df = extract_summary_matrix(model_rds)
if "lv" in summary_df.columns and "LV" not in summary_df.columns:
    summary_df = summary_df.rename(columns={"lv": "LV"})

print(f"lv_matrix : {lv_matrix.shape}  (samples × LVs)")
print(f"summary_df: {summary_df.shape}")
print(f"summary columns: {summary_df.columns.tolist()}")
lv_matrix.head()

model_rds = None
B_df = None

import gc
gc.collect()

lv_matrix : (605614, 1728)  (samples × LVs)
summary_df: (9100, 7)
summary columns: ['pathway', 'LV', 'AUC', 'p_value', 'FDR', 'npos', 'nneg']


218

## ARCHS4 metadata & analysis helpers

Load sample-level metadata from the h5 file once, pre-extract the B matrix, and
define the `analyze(label, keyword)` helper that runs the full pipeline for any tissue or cell type.

In [4]:
h5_path = str(here("data/archs4/human_gene_v2.5.h5"))

def _decode(arr):
    return np.array([x.decode("utf-8", errors="replace") for x in arr])

with h5py.File(h5_path, "r") as f:
    geo_acc = _decode(f["meta/samples/geo_accession"][:])
    source  = _decode(f["meta/samples/source_name_ch1"][:])
    char    = _decode(f["meta/samples/characteristics_ch1"][:])
    sc_prob = f["meta/samples/singlecellprobability"][:]

# Pre-extract B matrix array for fast per-LV operations
B_arr   = lv_matrix.values
n_total = len(lv_matrix)
K_1pct  = max(1, int(round(n_total * TOP_PERC)))

print(f"Metadata loaded : {len(geo_acc):,} samples")
print(f"B matrix        : {B_arr.shape}  (samples × LVs)")
print(f"Top-1% window   : {K_1pct:,} samples")

Metadata loaded : 888,821 samples
B matrix        : (605614, 1728)  (samples × LVs)
Top-1% window   : 6,056 samples


In [5]:
def get_samples(keyword):
    """Return GSM IDs matching `keyword` in source or characteristics (bulk only)."""
    kw   = keyword.lower()
    mask = (
        (np.char.find(np.char.lower(source), kw) >= 0) |
        (np.char.find(np.char.lower(char),   kw) >= 0)
    ) & (sc_prob < SC_PROB_THRESH)
    gsms = geo_acc[mask].tolist()
    return [s for s in gsms if s in lv_matrix.index]


def compute_lv_enrichment(available_gsms):
    """Mann-Whitney enrichment of available_gsms across all LVs. Returns sorted DataFrame."""
    n_interest     = len(available_gsms)
    interest_idx   = np.array([lv_matrix.index.get_loc(s) for s in available_gsms])
    background_idx = np.setdiff1d(np.arange(n_total), interest_idx)
    i_vals         = B_arr[interest_idx, :]
    b_vals         = B_arr[background_idx, :]

    rows = []
    for j, lv in enumerate(lv_matrix.columns):
        col            = B_arr[:, j]
        threshold_1pct = np.partition(col, -K_1pct)[-K_1pct]
        n_in_top       = int((i_vals[:, j] >= threshold_1pct).sum())
        obs_prop       = n_in_top / n_interest
        enrich_ratio   = obs_prop / (K_1pct / n_total)

        sorted_col      = np.sort(col)
        ranks_from_top  = n_total - np.searchsorted(sorted_col, i_vals[:, j], side="left")
        median_rank_pct = float(np.median(ranks_from_top)) / n_total

        stat, pval = mannwhitneyu(i_vals[:, j], b_vals[:, j], alternative="greater")
        auc        = stat / (len(interest_idx) * len(background_idx))

        rows.append({
            "LV": lv, "n_in_top1pct": n_in_top,
            "prop_in_top1pct": round(obs_prop, 4),
            "enrich_ratio": round(enrich_ratio, 3),
            "median_rank_pct": round(median_rank_pct, 4),
            "AUC": auc, "pvalue": pval,
        })

    res = pd.DataFrame(rows)
    _, res["FDR"], _, _ = multipletests(res["pvalue"], method="fdr_bh")
    return res.sort_values("enrich_ratio", ascending=False)


def analyze(label, keyword):
    """Full pipeline: find samples → enrich LVs → display pathways.
    Returns the pathways DataFrame for the selected LVs."""
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_colwidth", None)

    available = get_samples(keyword)
    print(f"{'─'*64}")
    print(f"  {label}  (keyword: '{keyword}')  |  samples in model: {len(available):,}")

    if len(available) < MIN_SAMPLES:
        print(f"  ⚠ fewer than {MIN_SAMPLES} samples – skipping.")
        return None

    res_df   = compute_lv_enrichment(available)
    top_lvs  = res_df[
        (res_df["FDR"] < FDR_THRESH_LVS) &
        (res_df["enrich_ratio"] >= ENRICH_RATIO_THRESH) &
        (res_df["n_in_top1pct"] >= MIN_N_IN_TOP1PCT)
    ].copy()
    names    = top_lvs["LV"].tolist()
    expected = K_1pct * len(available) / n_total

    print(f"  Expected in top 1%: {expected:.1f}  |  Selected LVs: {len(top_lvs)}")
    display(top_lvs[["LV", "n_in_top1pct", "enrich_ratio",
                      "median_rank_pct", "AUC", "FDR"]].reset_index(drop=True))

    # pathways
    summ = summary_df[
        summary_df["LV"].isin(names) &
        (summary_df["AUC"] > AUC_THRESH_PATHWAYS) &
        (summary_df["FDR"] < FDR_THRESH_PATHWAYS)
    ]
    print("\n  Pathways:")
    if summ.empty:
        print("  (none)")
    else:
        display(summ[["LV", "pathway", "AUC", "FDR"]].sort_values(["LV", "FDR"]).reset_index(drop=True))

    return summ[["LV", "pathway", "AUC", "FDR"]].sort_values(["LV", "FDR"]).reset_index(drop=True)

## Tissues

In [6]:
adipose_pathways = analyze("Adipose", "adipose")

────────────────────────────────────────────────────────────────
  Adipose  (keyword: 'adipose')  |  samples in model: 4,000
  Expected in top 1%: 40.0  |  Selected LVs: 83


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1293,748,18.700,0.3631,0.604907,1.856156e-114
1,LV1659,683,17.075,0.4568,0.554474,4.625716e-32
2,LV680,681,17.025,0.4470,0.540987,8.799496e-19
3,LV1286,676,16.900,0.4093,0.585488,1.211858e-76
4,LV60,674,16.850,0.4466,0.561737,1.026145e-40
5,LV1033,652,16.300,0.4961,0.552450,7.767644e-30
6,LV648,604,15.100,0.3657,0.587005,2.554572e-79
7,LV1253,595,14.875,0.4144,0.583671,1.823488e-73
8,LV651,567,14.175,0.5134,0.540080,5.058537e-18
9,LV35,543,13.575,0.2391,0.639723,5.440358e-202



  Pathways:


,LV,pathway,AUC,FDR
0,LV1012,C2CP_REACTOME_ABC_TRANSPORTERS_IN_LIPID_HOMEOSTASIS,0.906881,5.141231e-02
1,LV1012,C2CP_REACTOME_BUTYROPHILIN_BTN_FAMILY_INTERACTIONS,0.940989,7.283398e-02
2,LV1103,C2CP_REACTOME_ANTIGEN_PROCESSING_UB_ATP_INDEPENDENT_PROTEASOMAL_DEGRADATION,0.871444,6.564056e-02
3,LV1253,C2CP_KEGG_MEDICUS_REFERENCE_CXCR4_GNAQ_PLCB_G_CALCINEURIN_SIGNALING_PATHWAY,0.818197,7.144923e-02
4,LV1286,C2CP_BIOCARTA_P53_PATHWAY,0.994411,1.676426e-02
5,LV1286,C2CP_KEGG_MEDICUS_VARIANT_AMPLIFIED_MDM2_TO_P21_CELL_CYCLE_G1_S,1.000000,5.605625e-02
6,LV1286,C2CP_BIOCARTA_HCMV_PATHWAY,0.880063,6.697323e-02
7,LV1286,C2CP_BIOCARTA_RB_PATHWAY,0.940767,8.163289e-02
8,LV1342,C2CP_WP_OVERVIEW_OF_PROINFLAMMATORY_AND_PROFIBROTIC_MEDIATORS,0.737190,1.758426e-02
9,LV1342,C2CP_WP_OVERVIEW_OF_NANOPARTICLE_EFFECTS,0.847604,9.399659e-02


In [7]:
adipose_pathways

,LV,pathway,AUC,FDR
0,LV1012,C2CP_REACTOME_ABC_TRANSPORTERS_IN_LIPID_HOMEOSTASIS,0.906881,5.141231e-02
1,LV1012,C2CP_REACTOME_BUTYROPHILIN_BTN_FAMILY_INTERACTIONS,0.940989,7.283398e-02
2,LV1103,C2CP_REACTOME_ANTIGEN_PROCESSING_UB_ATP_INDEPENDENT_PROTEASOMAL_DEGRADATION,0.871444,6.564056e-02
3,LV1253,C2CP_KEGG_MEDICUS_REFERENCE_CXCR4_GNAQ_PLCB_G_CALCINEURIN_SIGNALING_PATHWAY,0.818197,7.144923e-02
4,LV1286,C2CP_BIOCARTA_P53_PATHWAY,0.994411,1.676426e-02
5,LV1286,C2CP_KEGG_MEDICUS_VARIANT_AMPLIFIED_MDM2_TO_P21_CELL_CYCLE_G1_S,1.000000,5.605625e-02
6,LV1286,C2CP_BIOCARTA_HCMV_PATHWAY,0.880063,6.697323e-02
7,LV1286,C2CP_BIOCARTA_RB_PATHWAY,0.940767,8.163289e-02
8,LV1342,C2CP_WP_OVERVIEW_OF_PROINFLAMMATORY_AND_PROFIBROTIC_MEDIATORS,0.737190,1.758426e-02
9,LV1342,C2CP_WP_OVERVIEW_OF_NANOPARTICLE_EFFECTS,0.847604,9.399659e-02


In [8]:
adrenal_gland_pathways = analyze("Adrenal gland", "adrenal gland")

────────────────────────────────────────────────────────────────
  Adrenal gland  (keyword: 'adrenal gland')  |  samples in model: 30
  Expected in top 1%: 0.3  |  Selected LVs: 0


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR



  Pathways:
  (none)


In [9]:
adrenal_gland_pathways

,LV,pathway,AUC,FDR


In [10]:
artery_pathways = analyze("Artery", "artery")

────────────────────────────────────────────────────────────────
  Artery  (keyword: 'artery')  |  samples in model: 697
  Expected in top 1%: 7.0  |  Selected LVs: 32


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1422,44,6.313,0.3700,0.592640,3.094262e-15
1,LV791,43,6.169,0.4295,0.538962,1.380842e-03
2,LV864,43,6.169,0.4406,0.546636,1.148206e-04
3,LV49,37,5.309,0.4307,0.553960,7.101209e-06
4,LV427,35,5.022,0.4538,0.544796,2.100888e-04
5,LV1602,34,4.878,0.4708,0.527620,2.734748e-02
6,LV689,33,4.735,0.4500,0.534449,5.115529e-03
7,LV1423,31,4.448,0.4878,0.532463,8.597787e-03
8,LV68,30,4.304,0.3868,0.587361,1.103939e-13
9,LV467,30,4.304,0.4580,0.539578,1.147126e-03



  Pathways:


,LV,pathway,AUC,FDR
0,LV1197,C2CP_KEGG_PHENYLALANINE_METABOLISM,0.898422,0.054814
1,LV217,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_ACTH_CORTISOL_SIGNALING_PATHWAY,0.993649,0.012122
2,LV217,C2CP_REACTOME_NUCLEAR_RECEPTOR_TRANSCRIPTION_PATHWAY,0.791728,0.016094
3,LV217,C2CP_WP_WHITE_FAT_CELL_DIFFERENTIATION,0.835253,0.016486
4,LV217,C2CP_BIOCARTA_PCAF_PATHWAY,0.964570,0.058881
5,LV217,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_ATP2B3_TO_ANGIOTENSIN_ALDOSTERONE_SIGNALING_PATHWAY,0.865324,0.066117
6,LV399,C2CP_REACTOME_ACYL_CHAIN_REMODELLING_OF_PG,1.000000,0.058193
7,LV44,C2CP_REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION,0.927785,0.000003
8,LV44,C2CP_REACTOME_ACTIVATED_PKN1_STIMULATES_TRANSCRIPTION_OF_AR_ANDROGEN_RECEPTOR_REGULATED_GENES_KLK2_AND_KLK3,0.921967,0.000012
9,LV44,C2CP_REACTOME_REPLACEMENT_OF_PROTAMINES_BY_NUCLEOSOMES_IN_THE_MALE_PRONUCLEUS,0.975729,0.000406


In [11]:
artery_pathways

,LV,pathway,AUC,FDR
0,LV1197,C2CP_KEGG_PHENYLALANINE_METABOLISM,0.898422,0.054814
1,LV217,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PRKAR1A_TO_ACTH_CORTISOL_SIGNALING_PATHWAY,0.993649,0.012122
2,LV217,C2CP_REACTOME_NUCLEAR_RECEPTOR_TRANSCRIPTION_PATHWAY,0.791728,0.016094
3,LV217,C2CP_WP_WHITE_FAT_CELL_DIFFERENTIATION,0.835253,0.016486
4,LV217,C2CP_BIOCARTA_PCAF_PATHWAY,0.964570,0.058881
5,LV217,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_ATP2B3_TO_ANGIOTENSIN_ALDOSTERONE_SIGNALING_PATHWAY,0.865324,0.066117
6,LV399,C2CP_REACTOME_ACYL_CHAIN_REMODELLING_OF_PG,1.000000,0.058193
7,LV44,C2CP_REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION,0.927785,0.000003
8,LV44,C2CP_REACTOME_ACTIVATED_PKN1_STIMULATES_TRANSCRIPTION_OF_AR_ANDROGEN_RECEPTOR_REGULATED_GENES_KLK2_AND_KLK3,0.921967,0.000012
9,LV44,C2CP_REACTOME_REPLACEMENT_OF_PROTAMINES_BY_NUCLEOSOMES_IN_THE_MALE_PRONUCLEUS,0.975729,0.000406


In [12]:
bladder_pathways = analyze("Bladder", "bladder")

────────────────────────────────────────────────────────────────
  Bladder  (keyword: 'bladder')  |  samples in model: 1,227
  Expected in top 1%: 12.3  |  Selected LVs: 50


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV727,114,9.291,0.4325,0.543289,1.074313e-06
1,LV14,98,7.987,0.4229,0.551838,4.325186e-09
2,LV1519,93,7.580,0.4220,0.558912,2.158713e-11
3,LV345,68,5.542,0.4643,0.538076,1.966342e-05
4,LV413,60,4.890,0.3714,0.597865,1.589010e-29
5,LV218,57,4.646,0.4293,0.568139,8.407296e-15
6,LV1470,52,4.238,0.4645,0.544152,6.649924e-07
7,LV731,49,3.994,0.3508,0.592866,9.222242e-27
8,LV1586,48,3.912,0.4734,0.528322,1.789913e-03
9,LV1373,47,3.831,0.4473,0.535607,6.753606e-05



  Pathways:


,LV,pathway,AUC,FDR
0,LV1103,C2CP_REACTOME_ANTIGEN_PROCESSING_UB_ATP_INDEPENDENT_PROTEASOMAL_DEGRADATION,0.871444,6.564056e-02
1,LV1373,C2CP_REACTOME_DEFECTIVE_EXT2_CAUSES_EXOSTOSES_2,0.942138,7.958803e-02
2,LV14,C2CP_KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISOME_DISASSEMBLY,0.941648,3.833069e-02
3,LV1519,C2CP_WP_ELECTRON_TRANSPORT_CHAIN_OXPHOS_SYSTEM_IN_MITOCHONDRIA,0.978760,1.362999e-11
4,LV1519,C2CP_WP_OXIDATIVE_PHOSPHORYLATION,1.000000,5.113629e-08
5,LV1519,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PINK1_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,0.999229,4.900400e-06
6,LV1519,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_SNCA_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,0.995800,5.927314e-06
7,LV1519,C2CP_KEGG_MEDICUS_REFERENCE_ELECTRON_TRANSFER_IN_COMPLEX_I,0.994851,2.592617e-05
8,LV1519,C2CP_REACTOME_FORMATION_OF_ATP_BY_CHEMIOSMOTIC_COUPLING,0.997951,5.321553e-03
9,LV1519,C2CP_KEGG_MEDICUS_REFERENCE_ELECTRON_TRANSFER_IN_COMPLEX_III,0.982136,6.564056e-02


In [13]:
bladder_pathways

,LV,pathway,AUC,FDR
0,LV1103,C2CP_REACTOME_ANTIGEN_PROCESSING_UB_ATP_INDEPENDENT_PROTEASOMAL_DEGRADATION,0.871444,6.564056e-02
1,LV1373,C2CP_REACTOME_DEFECTIVE_EXT2_CAUSES_EXOSTOSES_2,0.942138,7.958803e-02
2,LV14,C2CP_KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISOME_DISASSEMBLY,0.941648,3.833069e-02
3,LV1519,C2CP_WP_ELECTRON_TRANSPORT_CHAIN_OXPHOS_SYSTEM_IN_MITOCHONDRIA,0.978760,1.362999e-11
4,LV1519,C2CP_WP_OXIDATIVE_PHOSPHORYLATION,1.000000,5.113629e-08
5,LV1519,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_PINK1_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,0.999229,4.900400e-06
6,LV1519,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_SNCA_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,0.995800,5.927314e-06
7,LV1519,C2CP_KEGG_MEDICUS_REFERENCE_ELECTRON_TRANSFER_IN_COMPLEX_I,0.994851,2.592617e-05
8,LV1519,C2CP_REACTOME_FORMATION_OF_ATP_BY_CHEMIOSMOTIC_COUPLING,0.997951,5.321553e-03
9,LV1519,C2CP_KEGG_MEDICUS_REFERENCE_ELECTRON_TRANSFER_IN_COMPLEX_III,0.982136,6.564056e-02


In [14]:
blood_vessel_pathways = analyze("Blood vessel", "blood vessel")

────────────────────────────────────────────────────────────────
  Blood vessel  (keyword: 'blood vessel')  |  samples in model: 80
  Expected in top 1%: 0.8  |  Selected LVs: 0


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR



  Pathways:
  (none)


In [15]:
blood_vessel_pathways

,LV,pathway,AUC,FDR


In [16]:
brain_pathways = analyze("Brain", "brain")

────────────────────────────────────────────────────────────────
  Brain  (keyword: 'brain')  |  samples in model: 8,005
  Expected in top 1%: 80.0  |  Selected LVs: 31


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1016,455,5.684,0.4334,0.543593,9.334624e-40
1,LV142,435,5.434,0.4397,0.539045,3.650415e-32
2,LV259,301,3.760,0.4542,0.531959,4.376796e-22
3,LV1272,296,3.698,0.4429,0.538533,2.196889e-31
4,LV1343,283,3.535,0.4370,0.546368,1.072498e-44
5,LV125,265,3.311,0.4609,0.532636,5.667799e-23
6,LV1496,264,3.298,0.4719,0.524908,5.339152e-14
7,LV265,260,3.248,0.4549,0.530038,1.078941e-19
8,LV874,259,3.236,0.4591,0.529333,7.608401e-19
9,LV762,251,3.136,0.4660,0.516032,1.459544e-06



  Pathways:


,LV,pathway,AUC,FDR
0,LV1016,C2CP_REACTOME_DOPAMINE_NEUROTRANSMITTER_RELEASE_CYCLE,0.795684,0.093985
1,LV1024,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_MYOEPITHELIAL_CELLS,0.992725,0.004919
2,LV1024,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_LUMINAL_EPITHELIAL_CELLS,0.807711,0.087594
3,LV1024,C2CP_REACTOME_FORMATION_OF_THE_CORNIFIED_ENVELOPE,0.711649,0.096260
4,LV11,C2CP_WP_BIOMARKERS_FOR_UREA_CYCLE_DISORDERS,0.998419,0.055862
5,LV11,C2CP_KEGG_MEDICUS_REFERENCE_BILE_ACID_BIOSYNTHESIS,0.987243,0.057616
6,LV11,C2CP_WP_ANDROGEN_BIOSYNTHESIS,0.969225,0.063770
7,LV11,C2CP_REACTOME_SYNTHESIS_OF_BILE_ACIDS_AND_BILE_SALTS_VIA_24_HYDROXYCHOLESTEROL,0.964482,0.066117
8,LV11,C2CP_WP_FARNESOID_X_RECEPTOR_PATHWAY,0.959876,0.068223
9,LV11,C2CP_WP_CHOLESTASIS,0.844082,0.093431


In [17]:
brain_pathways

,LV,pathway,AUC,FDR
0,LV1016,C2CP_REACTOME_DOPAMINE_NEUROTRANSMITTER_RELEASE_CYCLE,0.795684,0.093985
1,LV1024,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_MYOEPITHELIAL_CELLS,0.992725,0.004919
2,LV1024,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_LUMINAL_EPITHELIAL_CELLS,0.807711,0.087594
3,LV1024,C2CP_REACTOME_FORMATION_OF_THE_CORNIFIED_ENVELOPE,0.711649,0.096260
4,LV11,C2CP_WP_BIOMARKERS_FOR_UREA_CYCLE_DISORDERS,0.998419,0.055862
5,LV11,C2CP_KEGG_MEDICUS_REFERENCE_BILE_ACID_BIOSYNTHESIS,0.987243,0.057616
6,LV11,C2CP_WP_ANDROGEN_BIOSYNTHESIS,0.969225,0.063770
7,LV11,C2CP_REACTOME_SYNTHESIS_OF_BILE_ACIDS_AND_BILE_SALTS_VIA_24_HYDROXYCHOLESTEROL,0.964482,0.066117
8,LV11,C2CP_WP_FARNESOID_X_RECEPTOR_PATHWAY,0.959876,0.068223
9,LV11,C2CP_WP_CHOLESTASIS,0.844082,0.093431


In [18]:
breast_pathways = analyze("Breast", "breast")

────────────────────────────────────────────────────────────────
  Breast  (keyword: 'breast')  |  samples in model: 17,357
  Expected in top 1%: 173.6  |  Selected LVs: 16


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1204,573,3.301,0.4618,0.528698,1.943191e-36
1,LV933,556,3.203,0.4825,0.519292,3.216391e-17
2,LV22,554,3.192,0.4921,0.512444,6.525862e-08
3,LV1091,547,3.152,0.4574,0.530265,2.901584e-40
4,LV739,520,2.996,0.4689,0.522641,3.259658e-23
5,LV1044,452,2.604,0.4644,0.523915,1.012077e-25
6,LV1630,441,2.541,0.4785,0.516020,2.784221e-12
7,LV734,413,2.379,0.4709,0.520815,8.098511e-20
8,LV1651,412,2.374,0.4668,0.520280,7.009986e-19
9,LV646,408,2.351,0.4881,0.506997,2.628510e-03



  Pathways:


,LV,pathway,AUC,FDR
0,LV1091,C2CP_WP_CILIOPATHIES,0.854116,2.023145e-11
1,LV1091,C2CP_WP_GENES_RELATED_TO_PRIMARY_CILIUM_DEVELOPMENT_BASED_ON_CRISPR,0.839100,1.530208e-06
2,LV1091,C2CP_REACTOME_INTRAFLAGELLAR_TRANSPORT,0.856616,1.655652e-03
3,LV156,C2CP_REACTOME_REGULATION_OF_NPAS4_GENE_EXPRESSION,0.989290,6.100384e-02
4,LV156,C2CP_REACTOME_REGULATION_OF_MITF_M_DEPENDENT_GENES_INVOLVED_IN_APOPTOSIS,0.866391,8.326865e-02
5,LV22,C2CP_WP_CILIOPATHIES,0.845144,1.117994e-10
6,LV253,C2CP_KEGG_MEDICUS_ENV_FACTOR_DCE_TO_DNA_ADDUCTS,0.998822,1.767292e-02
7,LV734,C2CP_KEGG_MEDICUS_ENV_FACTOR_TCDD_TO_AHR_SIGNALING_PATHWAY,1.000000,1.834856e-02
8,LV739,C2CP_KEGG_GLYCOSPHINGOLIPID_BIOSYNTHESIS_LACTO_AND_NEOLACTO_SERIES,0.921836,2.309212e-02
9,LV739,C2CP_REACTOME_BLOOD_GROUP_SYSTEMS_BIOSYNTHESIS,0.967580,3.131583e-02


In [19]:
breast_pathways

,LV,pathway,AUC,FDR
0,LV1091,C2CP_WP_CILIOPATHIES,0.854116,2.023145e-11
1,LV1091,C2CP_WP_GENES_RELATED_TO_PRIMARY_CILIUM_DEVELOPMENT_BASED_ON_CRISPR,0.839100,1.530208e-06
2,LV1091,C2CP_REACTOME_INTRAFLAGELLAR_TRANSPORT,0.856616,1.655652e-03
3,LV156,C2CP_REACTOME_REGULATION_OF_NPAS4_GENE_EXPRESSION,0.989290,6.100384e-02
4,LV156,C2CP_REACTOME_REGULATION_OF_MITF_M_DEPENDENT_GENES_INVOLVED_IN_APOPTOSIS,0.866391,8.326865e-02
5,LV22,C2CP_WP_CILIOPATHIES,0.845144,1.117994e-10
6,LV253,C2CP_KEGG_MEDICUS_ENV_FACTOR_DCE_TO_DNA_ADDUCTS,0.998822,1.767292e-02
7,LV734,C2CP_KEGG_MEDICUS_ENV_FACTOR_TCDD_TO_AHR_SIGNALING_PATHWAY,1.000000,1.834856e-02
8,LV739,C2CP_KEGG_GLYCOSPHINGOLIPID_BIOSYNTHESIS_LACTO_AND_NEOLACTO_SERIES,0.921836,2.309212e-02
9,LV739,C2CP_REACTOME_BLOOD_GROUP_SYSTEMS_BIOSYNTHESIS,0.967580,3.131583e-02


In [20]:
cervix_pathways = analyze("Cervix", "cervix")

────────────────────────────────────────────────────────────────
  Cervix  (keyword: 'cervix')  |  samples in model: 681
  Expected in top 1%: 6.8  |  Selected LVs: 44


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1466,54,7.930,0.4512,0.539507,1.424746e-03
1,LV568,54,7.930,0.3157,0.597170,7.117555e-16
2,LV1423,40,5.874,0.4021,0.547951,9.313039e-05
3,LV1502,38,5.580,0.4644,0.554731,6.650593e-06
4,LV891,34,4.993,0.4091,0.556018,4.001122e-06
5,LV971,33,4.846,0.4586,0.546971,1.292233e-04
6,LV1156,32,4.699,0.4061,0.561285,4.089615e-07
7,LV1117,31,4.552,0.4302,0.563521,1.427284e-07
8,LV1452,30,4.405,0.4519,0.535493,4.495844e-03
9,LV1362,28,4.112,0.4055,0.543579,4.068911e-04



  Pathways:


,LV,pathway,AUC,FDR
0,LV1028,C2CP_WP_METHIONINE_METABOLISM_LEADING_TO_SULFUR_AMINO_ACIDS_AND_RELATED_DISORDERS,0.963774,7.144923e-02
1,LV114,C2CP_REACTOME_KILLING_MECHANISMS,0.933013,9.130877e-02
2,LV1145,C2CP_KEGG_MEDICUS_REFERENCE_MDM2_P21_CELL_CYCLE_G1_S_N00536,0.965898,7.039529e-02
3,LV1225,C2CP_KEGG_MEDICUS_REFERENCE_BMP_SIGNALING_PATHWAY,0.868451,7.187528e-02
4,LV1269,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_SOD1_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.986098,3.313303e-05
5,LV1269,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_HTT_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.999766,7.165267e-05
6,LV1269,C2CP_KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.997593,7.758650e-05
7,LV1269,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.986418,1.159550e-04
8,LV1269,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.982703,1.355744e-04
9,LV1269,C2CP_KEGG_MEDICUS_REFERENCE_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.972103,2.088242e-04


In [21]:
cervix_pathways

,LV,pathway,AUC,FDR
0,LV1028,C2CP_WP_METHIONINE_METABOLISM_LEADING_TO_SULFUR_AMINO_ACIDS_AND_RELATED_DISORDERS,0.963774,7.144923e-02
1,LV114,C2CP_REACTOME_KILLING_MECHANISMS,0.933013,9.130877e-02
2,LV1145,C2CP_KEGG_MEDICUS_REFERENCE_MDM2_P21_CELL_CYCLE_G1_S_N00536,0.965898,7.039529e-02
3,LV1225,C2CP_KEGG_MEDICUS_REFERENCE_BMP_SIGNALING_PATHWAY,0.868451,7.187528e-02
4,LV1269,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_SOD1_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.986098,3.313303e-05
5,LV1269,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_HTT_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.999766,7.165267e-05
6,LV1269,C2CP_KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.997593,7.758650e-05
7,LV1269,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.986418,1.159550e-04
8,LV1269,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.982703,1.355744e-04
9,LV1269,C2CP_KEGG_MEDICUS_REFERENCE_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.972103,2.088242e-04


In [22]:
colon_pathways = analyze("Colon", "colon")

────────────────────────────────────────────────────────────────
  Colon  (keyword: 'colon')  |  samples in model: 5,768
  Expected in top 1%: 57.7  |  Selected LVs: 24


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1723,207,3.589,0.4753,0.513889,9.005612e-04
1,LV1218,201,3.485,0.4658,0.517250,3.016837e-05
2,LV1048,200,3.467,0.4601,0.527359,1.700139e-11
3,LV8,198,3.433,0.4625,0.518875,4.805176e-06
4,LV978,193,3.346,0.4599,0.519449,2.423844e-06
5,LV1409,171,2.965,0.4732,0.518699,5.908984e-06
6,LV520,151,2.618,0.4830,0.512915,2.169933e-03
7,LV349,143,2.479,0.4826,0.517469,2.370567e-05
8,LV1416,138,2.393,0.4837,0.513984,8.266831e-04
9,LV1652,134,2.323,0.4753,0.521210,2.326445e-07



  Pathways:


,LV,pathway,AUC,FDR
0,LV1048,C2CP_REACTOME_APOPTOSIS_INDUCED_DNA_FRAGMENTATION,0.940064,7.291895e-02
1,LV1218,C2CP_REACTOME_BASE_EXCISION_REPAIR_AP_SITE_FORMATION,0.975499,1.336140e-06
2,LV1218,C2CP_REACTOME_RECOGNITION_AND_ASSOCIATION_OF_DNA_GLYCOSYLASE_WITH_SITE_CONTAINING_AN_AFFECTED_PURINE,0.972102,2.262292e-05
3,LV1218,C2CP_WP_TELOMERE_END_PACKAGING_AND_NEURODEVELOPMENTAL_DISORDERS,0.883677,2.164796e-03
4,LV1218,C2CP_REACTOME_REPLACEMENT_OF_PROTAMINES_BY_NUCLEOSOMES_IN_THE_MALE_PRONUCLEUS,0.914875,4.702992e-03
5,LV1218,C2CP_WP_TESSADORIBICKNELLVAN_HAAFTEN_SYNDROME_VARIANTS_NUCLEOSOME_ASSEMBLY,1.000000,1.977278e-02
6,LV1218,C2CP_REACTOME_DEPOSITION_OF_NEW_CENPA_CONTAINING_NUCLEOSOMES_AT_THE_CENTROMERE,0.721876,3.503176e-02
7,LV1218,C2CP_REACTOME_DNA_METHYLATION,0.701250,7.184344e-02
8,LV1218,C2CP_REACTOME_CONDENSATION_OF_PROPHASE_CHROMOSOMES,0.668963,9.986911e-02
9,LV1416,C2CP_REACTOME_BIOTIN_TRANSPORT_AND_METABOLISM,0.960350,7.447777e-02


In [23]:
colon_pathways

,LV,pathway,AUC,FDR
0,LV1048,C2CP_REACTOME_APOPTOSIS_INDUCED_DNA_FRAGMENTATION,0.940064,7.291895e-02
1,LV1218,C2CP_REACTOME_BASE_EXCISION_REPAIR_AP_SITE_FORMATION,0.975499,1.336140e-06
2,LV1218,C2CP_REACTOME_RECOGNITION_AND_ASSOCIATION_OF_DNA_GLYCOSYLASE_WITH_SITE_CONTAINING_AN_AFFECTED_PURINE,0.972102,2.262292e-05
3,LV1218,C2CP_WP_TELOMERE_END_PACKAGING_AND_NEURODEVELOPMENTAL_DISORDERS,0.883677,2.164796e-03
4,LV1218,C2CP_REACTOME_REPLACEMENT_OF_PROTAMINES_BY_NUCLEOSOMES_IN_THE_MALE_PRONUCLEUS,0.914875,4.702992e-03
5,LV1218,C2CP_WP_TESSADORIBICKNELLVAN_HAAFTEN_SYNDROME_VARIANTS_NUCLEOSOME_ASSEMBLY,1.000000,1.977278e-02
6,LV1218,C2CP_REACTOME_DEPOSITION_OF_NEW_CENPA_CONTAINING_NUCLEOSOMES_AT_THE_CENTROMERE,0.721876,3.503176e-02
7,LV1218,C2CP_REACTOME_DNA_METHYLATION,0.701250,7.184344e-02
8,LV1218,C2CP_REACTOME_CONDENSATION_OF_PROPHASE_CHROMOSOMES,0.668963,9.986911e-02
9,LV1416,C2CP_REACTOME_BIOTIN_TRANSPORT_AND_METABOLISM,0.960350,7.447777e-02


In [24]:
esophagus_pathways = analyze("Esophagus", "esophagus")

────────────────────────────────────────────────────────────────
  Esophagus  (keyword: 'esophagus')  |  samples in model: 650
  Expected in top 1%: 6.5  |  Selected LVs: 57


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV291,60,9.231,0.4335,0.537580,2.094774e-03
1,LV146,50,7.692,0.4163,0.558250,1.264360e-06
2,LV690,46,7.077,0.4441,0.559454,7.504734e-07
3,LV1229,44,6.769,0.3967,0.583665,2.686565e-12
4,LV1351,44,6.769,0.4688,0.531330,1.074605e-02
5,LV1344,40,6.154,0.4125,0.578666,4.577100e-11
6,LV21,40,6.154,0.3959,0.563331,1.289232e-07
7,LV111,39,6.000,0.4235,0.536407,2.925917e-03
8,LV1706,37,5.692,0.3830,0.571889,1.875738e-09
9,LV1542,36,5.539,0.4627,0.542799,4.451802e-04



  Pathways:


,LV,pathway,AUC,FDR
0,LV1114,C2CP_WP_STRIATED_MUSCLE_CONTRACTION_PATHWAY,0.971312,9.061594e-04
1,LV1114,C2CP_REACTOME_STRIATED_MUSCLE_CONTRACTION,0.986739,2.108424e-03
2,LV1365,C2CP_REACTOME_O_LINKED_GLYCOSYLATION,0.761779,5.595218e-04
3,LV1365,C2CP_WP_MATRIX_METALLOPROTEINASES,0.919316,2.171194e-02
4,LV1365,C2CP_WP_MONOAMINE_GPCRS,1.000000,5.756743e-02
5,LV1365,C2CP_REACTOME_DISEASES_ASSOCIATED_WITH_O_GLYCOSYLATION_OF_PROTEINS,0.715655,6.290342e-02
6,LV1389,C2CP_KEGG_LYSOSOME,0.831486,2.778235e-07
7,LV1389,C2CP_REACTOME_REGULATION_OF_MITF_M_DEPENDENT_GENES_INVOLVED_IN_LYSOSOME_BIOGENESIS_AND_AUTOPHAGY,0.983702,1.822399e-02
8,LV1389,C2CP_WP_DEGRADATION_PATHWAY_OF_SPHINGOLIPIDS_INCLUDING_DISEASES,0.998904,5.568239e-02
9,LV149,C2CP_REACTOME_CS_DS_DEGRADATION,0.858506,8.731811e-02


In [25]:
esophagus_pathways

,LV,pathway,AUC,FDR
0,LV1114,C2CP_WP_STRIATED_MUSCLE_CONTRACTION_PATHWAY,0.971312,9.061594e-04
1,LV1114,C2CP_REACTOME_STRIATED_MUSCLE_CONTRACTION,0.986739,2.108424e-03
2,LV1365,C2CP_REACTOME_O_LINKED_GLYCOSYLATION,0.761779,5.595218e-04
3,LV1365,C2CP_WP_MATRIX_METALLOPROTEINASES,0.919316,2.171194e-02
4,LV1365,C2CP_WP_MONOAMINE_GPCRS,1.000000,5.756743e-02
5,LV1365,C2CP_REACTOME_DISEASES_ASSOCIATED_WITH_O_GLYCOSYLATION_OF_PROTEINS,0.715655,6.290342e-02
6,LV1389,C2CP_KEGG_LYSOSOME,0.831486,2.778235e-07
7,LV1389,C2CP_REACTOME_REGULATION_OF_MITF_M_DEPENDENT_GENES_INVOLVED_IN_LYSOSOME_BIOGENESIS_AND_AUTOPHAGY,0.983702,1.822399e-02
8,LV1389,C2CP_WP_DEGRADATION_PATHWAY_OF_SPHINGOLIPIDS_INCLUDING_DISEASES,0.998904,5.568239e-02
9,LV149,C2CP_REACTOME_CS_DS_DEGRADATION,0.858506,8.731811e-02


In [26]:
heart_pathways = analyze("Heart", "heart")

────────────────────────────────────────────────────────────────
  Heart  (keyword: 'heart')  |  samples in model: 1,959
  Expected in top 1%: 19.6  |  Selected LVs: 96


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV82,177,9.035,0.3623,0.580442,6.603973e-32
1,LV104,117,5.973,0.4525,0.535464,3.407298e-07
2,LV351,105,5.360,0.5016,0.518120,1.140058e-02
3,LV967,100,5.105,0.4510,0.539211,1.586296e-08
4,LV1493,98,5.003,0.4643,0.534760,5.682765e-07
5,LV1282,95,4.850,0.4728,0.540943,3.413076e-09
6,LV882,95,4.850,0.4607,0.524598,5.108630e-04
7,LV1562,92,4.696,0.4654,0.525421,3.214283e-04
8,LV1166,88,4.492,0.4426,0.548019,3.710843e-12
9,LV162,86,4.390,0.4510,0.534230,8.321574e-07



  Pathways:


,LV,pathway,AUC,FDR
0,LV1024,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_MYOEPITHELIAL_CELLS,0.992725,0.004919
1,LV1024,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_LUMINAL_EPITHELIAL_CELLS,0.807711,0.087594
2,LV1024,C2CP_REACTOME_FORMATION_OF_THE_CORNIFIED_ENVELOPE,0.711649,0.096260
3,LV105,C2CP_WP_SULFATION_BIOTRANSFORMATION_REACTION,0.996219,0.015614
4,LV105,C2CP_WP_TAMOXIFEN_METABOLISM,0.999700,0.055682
5,LV1057,C2CP_BIOCARTA_VITCB_PATHWAY,0.934916,0.093600
6,LV1166,C2CP_BIOCARTA_COMP_PATHWAY,0.960698,0.026963
7,LV1166,C2CP_BIOCARTA_CLASSIC_PATHWAY,0.988744,0.057588
8,LV119,C2CP_WP_NOTCH1_REGULATION_OF_ENDOTHELIAL_CELL_CALCIFICATION,0.991041,0.018023
9,LV119,C2CP_WP_OLIGODENDROCYTE_DEVELOPMENT,0.801646,0.028716


In [27]:
heart_pathways

,LV,pathway,AUC,FDR
0,LV1024,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_MYOEPITHELIAL_CELLS,0.992725,0.004919
1,LV1024,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_LUMINAL_EPITHELIAL_CELLS,0.807711,0.087594
2,LV1024,C2CP_REACTOME_FORMATION_OF_THE_CORNIFIED_ENVELOPE,0.711649,0.096260
3,LV105,C2CP_WP_SULFATION_BIOTRANSFORMATION_REACTION,0.996219,0.015614
4,LV105,C2CP_WP_TAMOXIFEN_METABOLISM,0.999700,0.055682
5,LV1057,C2CP_BIOCARTA_VITCB_PATHWAY,0.934916,0.093600
6,LV1166,C2CP_BIOCARTA_COMP_PATHWAY,0.960698,0.026963
7,LV1166,C2CP_BIOCARTA_CLASSIC_PATHWAY,0.988744,0.057588
8,LV119,C2CP_WP_NOTCH1_REGULATION_OF_ENDOTHELIAL_CELL_CALCIFICATION,0.991041,0.018023
9,LV119,C2CP_WP_OLIGODENDROCYTE_DEVELOPMENT,0.801646,0.028716


In [28]:
kidney_pathways = analyze("Kidney", "kidney")

────────────────────────────────────────────────────────────────
  Kidney  (keyword: 'kidney')  |  samples in model: 4,820
  Expected in top 1%: 48.2  |  Selected LVs: 23


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1458,141,2.925,0.4801,0.515349,7.476036e-04
1,LV1363,135,2.801,0.4818,0.520117,7.429937e-06
2,LV1235,135,2.801,0.4819,0.516869,2.055134e-04
3,LV1218,134,2.780,0.4825,0.514879,1.095508e-03
4,LV1516,122,2.531,0.4692,0.526813,1.445294e-09
5,LV112,122,2.531,0.4719,0.525565,8.571958e-09
6,LV1468,121,2.510,0.4860,0.509576,4.175956e-02
7,LV978,118,2.448,0.4662,0.524816,2.396947e-08
8,LV741,114,2.365,0.4565,0.527133,9.671337e-10
9,LV130,113,2.344,0.4845,0.509814,3.668733e-02



  Pathways:


,LV,pathway,AUC,FDR
0,LV1048,C2CP_REACTOME_APOPTOSIS_INDUCED_DNA_FRAGMENTATION,0.940064,7.291895e-02
1,LV1218,C2CP_REACTOME_BASE_EXCISION_REPAIR_AP_SITE_FORMATION,0.975499,1.336140e-06
2,LV1218,C2CP_REACTOME_RECOGNITION_AND_ASSOCIATION_OF_DNA_GLYCOSYLASE_WITH_SITE_CONTAINING_AN_AFFECTED_PURINE,0.972102,2.262292e-05
3,LV1218,C2CP_WP_TELOMERE_END_PACKAGING_AND_NEURODEVELOPMENTAL_DISORDERS,0.883677,2.164796e-03
4,LV1218,C2CP_REACTOME_REPLACEMENT_OF_PROTAMINES_BY_NUCLEOSOMES_IN_THE_MALE_PRONUCLEUS,0.914875,4.702992e-03
5,LV1218,C2CP_WP_TESSADORIBICKNELLVAN_HAAFTEN_SYNDROME_VARIANTS_NUCLEOSOME_ASSEMBLY,1.000000,1.977278e-02
6,LV1218,C2CP_REACTOME_DEPOSITION_OF_NEW_CENPA_CONTAINING_NUCLEOSOMES_AT_THE_CENTROMERE,0.721876,3.503176e-02
7,LV1218,C2CP_REACTOME_DNA_METHYLATION,0.701250,7.184344e-02
8,LV1218,C2CP_REACTOME_CONDENSATION_OF_PROPHASE_CHROMOSOMES,0.668963,9.986911e-02
9,LV1363,C2CP_REACTOME_EUKARYOTIC_TRANSLATION_ELONGATION,1.000000,8.684349e-12


In [29]:
kidney_pathways

,LV,pathway,AUC,FDR
0,LV1048,C2CP_REACTOME_APOPTOSIS_INDUCED_DNA_FRAGMENTATION,0.940064,7.291895e-02
1,LV1218,C2CP_REACTOME_BASE_EXCISION_REPAIR_AP_SITE_FORMATION,0.975499,1.336140e-06
2,LV1218,C2CP_REACTOME_RECOGNITION_AND_ASSOCIATION_OF_DNA_GLYCOSYLASE_WITH_SITE_CONTAINING_AN_AFFECTED_PURINE,0.972102,2.262292e-05
3,LV1218,C2CP_WP_TELOMERE_END_PACKAGING_AND_NEURODEVELOPMENTAL_DISORDERS,0.883677,2.164796e-03
4,LV1218,C2CP_REACTOME_REPLACEMENT_OF_PROTAMINES_BY_NUCLEOSOMES_IN_THE_MALE_PRONUCLEUS,0.914875,4.702992e-03
5,LV1218,C2CP_WP_TESSADORIBICKNELLVAN_HAAFTEN_SYNDROME_VARIANTS_NUCLEOSOME_ASSEMBLY,1.000000,1.977278e-02
6,LV1218,C2CP_REACTOME_DEPOSITION_OF_NEW_CENPA_CONTAINING_NUCLEOSOMES_AT_THE_CENTROMERE,0.721876,3.503176e-02
7,LV1218,C2CP_REACTOME_DNA_METHYLATION,0.701250,7.184344e-02
8,LV1218,C2CP_REACTOME_CONDENSATION_OF_PROPHASE_CHROMOSOMES,0.668963,9.986911e-02
9,LV1363,C2CP_REACTOME_EUKARYOTIC_TRANSLATION_ELONGATION,1.000000,8.684349e-12


In [30]:
liver_pathways = analyze("Liver", "liver")

────────────────────────────────────────────────────────────────
  Liver  (keyword: 'liver')  |  samples in model: 9,197
  Expected in top 1%: 92.0  |  Selected LVs: 18


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1434,763,8.296,0.4509,0.537629,1.005073e-33
1,LV303,744,8.090,0.4269,0.557874,1.067666e-78
2,LV15,738,8.025,0.4487,0.548030,3.106115e-54
3,LV19,720,7.829,0.4129,0.572685,5.892540e-124
4,LV298,442,4.806,0.4753,0.533055,2.222827e-26
5,LV1468,287,3.121,0.4883,0.507955,1.511532e-02
6,LV1420,285,3.099,0.4680,0.536547,6.544606e-32
7,LV13,227,2.468,0.4873,0.514297,7.058793e-06
8,LV1495,217,2.360,0.4470,0.534376,2.099654e-28
9,LV60,213,2.316,0.4916,0.517854,1.603373e-08



  Pathways:


,LV,pathway,AUC,FDR
0,LV1420,C2CP_REACTOME_MEIOTIC_RECOMBINATION,0.833417,9.408245e-05
1,LV1420,C2CP_REACTOME_INHIBITION_OF_DNA_RECOMBINATION_AT_TELOMERE,0.870496,1.057661e-04
2,LV1420,C2CP_REACTOME_TRANSCRIPTIONAL_REGULATION_BY_SMALL_RNAS,0.812408,5.318460e-04
3,LV1420,C2CP_REACTOME_DNA_METHYLATION,0.840944,8.864573e-04
4,LV1420,C2CP_REACTOME_RECOGNITION_AND_ASSOCIATION_OF_DNA_GLYCOSYLASE_WITH_SITE_CONTAINING_AN_AFFECTED_PURINE,0.849927,2.513417e-03
5,LV1420,C2CP_REACTOME_ACTIVATED_PKN1_STIMULATES_TRANSCRIPTION_OF_AR_ANDROGEN_RECEPTOR_REGULATED_GENES_KLK2_AND_KLK3,0.805951,3.771898e-03
6,LV1420,C2CP_REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION,0.760770,1.214457e-02
7,LV1434,C2CP_REACTOME_ASSEMBLY_OF_THE_ORC_COMPLEX_AT_THE_ORIGIN_OF_REPLICATION,0.967495,4.026426e-07
8,LV1434,C2CP_REACTOME_BASE_EXCISION_REPAIR,0.843248,3.386783e-05
9,LV1434,C2CP_WP_TELOMERE_END_PACKAGING_AND_NEURODEVELOPMENTAL_DISORDERS,0.981626,3.918928e-05


In [31]:
liver_pathways

,LV,pathway,AUC,FDR
0,LV1420,C2CP_REACTOME_MEIOTIC_RECOMBINATION,0.833417,9.408245e-05
1,LV1420,C2CP_REACTOME_INHIBITION_OF_DNA_RECOMBINATION_AT_TELOMERE,0.870496,1.057661e-04
2,LV1420,C2CP_REACTOME_TRANSCRIPTIONAL_REGULATION_BY_SMALL_RNAS,0.812408,5.318460e-04
3,LV1420,C2CP_REACTOME_DNA_METHYLATION,0.840944,8.864573e-04
4,LV1420,C2CP_REACTOME_RECOGNITION_AND_ASSOCIATION_OF_DNA_GLYCOSYLASE_WITH_SITE_CONTAINING_AN_AFFECTED_PURINE,0.849927,2.513417e-03
5,LV1420,C2CP_REACTOME_ACTIVATED_PKN1_STIMULATES_TRANSCRIPTION_OF_AR_ANDROGEN_RECEPTOR_REGULATED_GENES_KLK2_AND_KLK3,0.805951,3.771898e-03
6,LV1420,C2CP_REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION,0.760770,1.214457e-02
7,LV1434,C2CP_REACTOME_ASSEMBLY_OF_THE_ORC_COMPLEX_AT_THE_ORIGIN_OF_REPLICATION,0.967495,4.026426e-07
8,LV1434,C2CP_REACTOME_BASE_EXCISION_REPAIR,0.843248,3.386783e-05
9,LV1434,C2CP_WP_TELOMERE_END_PACKAGING_AND_NEURODEVELOPMENTAL_DISORDERS,0.981626,3.918928e-05


In [32]:
lung_pathways = analyze("Lung", "lung")

────────────────────────────────────────────────────────────────
  Lung  (keyword: 'lung')  |  samples in model: 12,053
  Expected in top 1%: 120.5  |  Selected LVs: 3


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1373,307,2.547,0.4828,0.506985,1.549307e-02
1,LV186,265,2.199,0.4888,0.510744,1.501030e-04
2,LV222,248,2.058,0.4763,0.517431,4.955622e-10



  Pathways:


,LV,pathway,AUC,FDR
0,LV1373,C2CP_REACTOME_DEFECTIVE_EXT2_CAUSES_EXOSTOSES_2,0.942138,0.079588


In [33]:
lung_pathways

,LV,pathway,AUC,FDR
0,LV1373,C2CP_REACTOME_DEFECTIVE_EXT2_CAUSES_EXOSTOSES_2,0.942138,0.079588


In [34]:
muscle_pathways = analyze("Muscle", "muscle")

────────────────────────────────────────────────────────────────
  Muscle  (keyword: 'muscle')  |  samples in model: 4,572
  Expected in top 1%: 45.7  |  Selected LVs: 34


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV147,216,4.725,0.4600,0.531013,6.850254e-12
1,LV732,212,4.637,0.4566,0.526167,8.835424e-09
2,LV1188,161,3.522,0.4760,0.515595,7.577196e-04
3,LV94,145,3.172,0.4793,0.517178,1.963966e-04
4,LV1145,143,3.128,0.4457,0.525753,1.489558e-08
5,LV418,143,3.128,0.4631,0.522528,7.918254e-07
6,LV1103,139,3.040,0.4650,0.528239,5.084648e-10
7,LV13,128,2.800,0.4447,0.534494,1.676039e-14
8,LV1674,128,2.800,0.4680,0.519455,2.317620e-05
9,LV1388,112,2.450,0.4689,0.524940,4.141597e-08



  Pathways:


,LV,pathway,AUC,FDR
0,LV1103,C2CP_REACTOME_ANTIGEN_PROCESSING_UB_ATP_INDEPENDENT_PROTEASOMAL_DEGRADATION,0.871444,0.065641
1,LV1145,C2CP_KEGG_MEDICUS_REFERENCE_MDM2_P21_CELL_CYCLE_G1_S_N00536,0.965898,0.070395
2,LV1227,C2CP_WP_TRANSCRIPTIONAL_CASCADE_REGULATING_ADIPOGENESIS,0.984989,0.057418
3,LV1227,C2CP_WP_NAD_METABOLISM_SIRTUINS_AND_AGING,0.952929,0.070179
4,LV1300,C2CP_WP_STATIN_INHIBITION_OF_CHOLESTEROL_PRODUCTION,0.935820,0.017507
5,LV1300,C2CP_WP_FAMILIAL_HYPERLIPIDEMIA_TYPE_3,1.000000,0.058237
6,LV1300,C2CP_WP_FAMILIAL_HYPERLIPIDEMIA_TYPE_5,1.000000,0.058237
7,LV1412,C2CP_REACTOME_FIBRONECTIN_MATRIX_FORMATION,0.924274,0.016461
8,LV1412,C2CP_REACTOME_ATTACHMENT_OF_BACTERIA_TO_EPITHELIAL_CELLS,0.795323,0.094570
9,LV147,C2CP_REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION,0.673349,0.074990


In [35]:
muscle_pathways

,LV,pathway,AUC,FDR
0,LV1103,C2CP_REACTOME_ANTIGEN_PROCESSING_UB_ATP_INDEPENDENT_PROTEASOMAL_DEGRADATION,0.871444,0.065641
1,LV1145,C2CP_KEGG_MEDICUS_REFERENCE_MDM2_P21_CELL_CYCLE_G1_S_N00536,0.965898,0.070395
2,LV1227,C2CP_WP_TRANSCRIPTIONAL_CASCADE_REGULATING_ADIPOGENESIS,0.984989,0.057418
3,LV1227,C2CP_WP_NAD_METABOLISM_SIRTUINS_AND_AGING,0.952929,0.070179
4,LV1300,C2CP_WP_STATIN_INHIBITION_OF_CHOLESTEROL_PRODUCTION,0.935820,0.017507
5,LV1300,C2CP_WP_FAMILIAL_HYPERLIPIDEMIA_TYPE_3,1.000000,0.058237
6,LV1300,C2CP_WP_FAMILIAL_HYPERLIPIDEMIA_TYPE_5,1.000000,0.058237
7,LV1412,C2CP_REACTOME_FIBRONECTIN_MATRIX_FORMATION,0.924274,0.016461
8,LV1412,C2CP_REACTOME_ATTACHMENT_OF_BACTERIA_TO_EPITHELIAL_CELLS,0.795323,0.094570
9,LV147,C2CP_REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION,0.673349,0.074990


In [36]:
nerve_pathways = analyze("Nerve", "nerve")

────────────────────────────────────────────────────────────────
  Nerve  (keyword: 'nerve')  |  samples in model: 143
  Expected in top 1%: 1.4  |  Selected LVs: 22


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV132,50,34.966,0.1979,0.791178,3.687987e-31
1,LV509,44,30.770,0.1249,0.677494,1.673339e-12
2,LV512,43,30.071,0.3350,0.642430,1.694525e-08
3,LV1215,39,27.273,0.0840,0.688743,6.038280e-14
4,LV403,36,25.175,0.3691,0.619010,2.828425e-06
5,LV120,35,24.476,0.1532,0.707110,1.567605e-16
6,LV1528,31,21.679,0.1627,0.718195,3.740090e-18
7,LV1494,28,19.581,0.1355,0.678511,1.257412e-12
8,LV1063,28,19.581,0.3331,0.611247,1.273902e-05
9,LV78,28,19.581,0.4610,0.588809,5.509696e-04



  Pathways:


,LV,pathway,AUC,FDR
0,LV1109,C2CP_REACTOME_TRYPTOPHAN_CATABOLISM,0.985284,0.057616
1,LV120,C2CP_WP_ULCERATIVE_COLITIS_SIGNALING,0.986072,0.018224
2,LV120,C2CP_WP_IL23_INHIBITORS_IN_INFLAMMATORY_BOWEL_DISEASE,0.994238,0.056403
3,LV1215,C2CP_KEGG_ALLOGRAFT_REJECTION,1.000000,0.000334
4,LV1215,C2CP_KEGG_GRAFT_VERSUS_HOST_DISEASE,0.972925,0.000809
5,LV1215,C2CP_KEGG_ASTHMA,0.985244,0.007149
6,LV1215,C2CP_REACTOME_CHEMOKINE_RECEPTORS_BIND_CHEMOKINES,0.843191,0.007149
7,LV1215,C2CP_BIOCARTA_ASBCELL_PATHWAY,1.000000,0.058991
8,LV1577,C2CP_REACTOME_SYNTHESIS_OF_GLYCOSYLPHOSPHATIDYLINOSITOL_GPI,0.887161,0.066373
9,LV1667,C2CP_BIOCARTA_PLATELETAPP_PATHWAY,0.964891,0.058957


In [37]:
nerve_pathways

,LV,pathway,AUC,FDR
0,LV1109,C2CP_REACTOME_TRYPTOPHAN_CATABOLISM,0.985284,0.057616
1,LV120,C2CP_WP_ULCERATIVE_COLITIS_SIGNALING,0.986072,0.018224
2,LV120,C2CP_WP_IL23_INHIBITORS_IN_INFLAMMATORY_BOWEL_DISEASE,0.994238,0.056403
3,LV1215,C2CP_KEGG_ALLOGRAFT_REJECTION,1.000000,0.000334
4,LV1215,C2CP_KEGG_GRAFT_VERSUS_HOST_DISEASE,0.972925,0.000809
5,LV1215,C2CP_KEGG_ASTHMA,0.985244,0.007149
6,LV1215,C2CP_REACTOME_CHEMOKINE_RECEPTORS_BIND_CHEMOKINES,0.843191,0.007149
7,LV1215,C2CP_BIOCARTA_ASBCELL_PATHWAY,1.000000,0.058991
8,LV1577,C2CP_REACTOME_SYNTHESIS_OF_GLYCOSYLPHOSPHATIDYLINOSITOL_GPI,0.887161,0.066373
9,LV1667,C2CP_BIOCARTA_PLATELETAPP_PATHWAY,0.964891,0.058957


In [38]:
ovary_pathways = analyze("Ovary", "ovary")

────────────────────────────────────────────────────────────────
  Ovary  (keyword: 'ovary')  |  samples in model: 594
  Expected in top 1%: 5.9  |  Selected LVs: 37


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1006,83,13.973,0.4299,0.564533,3.511730e-07
1,LV886,76,12.795,0.3343,0.614018,6.225218e-20
2,LV312,69,11.616,0.3117,0.623273,3.476261e-23
3,LV725,65,10.943,0.4321,0.571059,1.915344e-08
4,LV1522,64,10.775,0.4616,0.554288,2.017842e-05
5,LV575,52,8.754,0.4442,0.542616,9.158954e-04
6,LV937,52,8.754,0.3555,0.584975,1.467149e-11
7,LV712,50,8.418,0.4071,0.572039,1.228605e-08
8,LV1061,44,7.408,0.3832,0.579188,3.443122e-10
9,LV1428,42,7.071,0.3893,0.587954,2.846236e-12



  Pathways:


,LV,pathway,AUC,FDR
0,LV1061,C2CP_KEGG_GLYCINE_SERINE_AND_THREONINE_METABOLISM,0.889486,0.015817
1,LV1061,C2CP_WP_DISORDERS_OF_FRUCTOSE_METABOLISM,1.000000,0.056403
2,LV1074,C2CP_WP_ANDROGEN_BIOSYNTHESIS,1.000000,0.058800
3,LV1114,C2CP_WP_STRIATED_MUSCLE_CONTRACTION_PATHWAY,0.971312,0.000906
4,LV1114,C2CP_REACTOME_STRIATED_MUSCLE_CONTRACTION,0.986739,0.002108
5,LV1138,C2CP_KEGG_STEROID_BIOSYNTHESIS,0.964529,0.023092
6,LV1148,C2CP_REACTOME_PRC2_METHYLATES_HISTONES_AND_DNA,0.902929,0.000007
7,LV1148,C2CP_REACTOME_DEFECTIVE_PYROPTOSIS,0.893290,0.000014
8,LV1148,C2CP_REACTOME_RECOGNITION_AND_ASSOCIATION_OF_DNA_GLYCOSYLASE_WITH_SITE_CONTAINING_AN_AFFECTED_PURINE,0.952526,0.000036
9,LV1148,C2CP_REACTOME_REGULATION_OF_ENDOGENOUS_RETROELEMENTS_BY_THE_HUMAN_SILENCING_HUB_HUSH_COMPLEX,0.841785,0.000438


In [39]:
ovary_pathways

,LV,pathway,AUC,FDR
0,LV1061,C2CP_KEGG_GLYCINE_SERINE_AND_THREONINE_METABOLISM,0.889486,0.015817
1,LV1061,C2CP_WP_DISORDERS_OF_FRUCTOSE_METABOLISM,1.000000,0.056403
2,LV1074,C2CP_WP_ANDROGEN_BIOSYNTHESIS,1.000000,0.058800
3,LV1114,C2CP_WP_STRIATED_MUSCLE_CONTRACTION_PATHWAY,0.971312,0.000906
4,LV1114,C2CP_REACTOME_STRIATED_MUSCLE_CONTRACTION,0.986739,0.002108
5,LV1138,C2CP_KEGG_STEROID_BIOSYNTHESIS,0.964529,0.023092
6,LV1148,C2CP_REACTOME_PRC2_METHYLATES_HISTONES_AND_DNA,0.902929,0.000007
7,LV1148,C2CP_REACTOME_DEFECTIVE_PYROPTOSIS,0.893290,0.000014
8,LV1148,C2CP_REACTOME_RECOGNITION_AND_ASSOCIATION_OF_DNA_GLYCOSYLASE_WITH_SITE_CONTAINING_AN_AFFECTED_PURINE,0.952526,0.000036
9,LV1148,C2CP_REACTOME_REGULATION_OF_ENDOGENOUS_RETROELEMENTS_BY_THE_HUMAN_SILENCING_HUB_HUSH_COMPLEX,0.841785,0.000438


In [40]:
pancreas_pathways = analyze("Pancreas", "pancreas")

────────────────────────────────────────────────────────────────
  Pancreas  (keyword: 'pancreas')  |  samples in model: 1,324
  Expected in top 1%: 13.2  |  Selected LVs: 55


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1262,85,6.420,0.4598,0.528107,1.892101e-03
1,LV919,60,4.532,0.4853,0.527406,2.489906e-03
2,LV936,59,4.456,0.4470,0.551143,5.756161e-09
3,LV584,58,4.381,0.4428,0.552321,2.721787e-09
4,LV30,54,4.079,0.4878,0.520635,2.538278e-02
5,LV620,53,4.003,0.4487,0.528014,1.960784e-03
6,LV1535,51,3.852,0.4379,0.538598,1.380308e-05
7,LV1481,50,3.777,0.4874,0.520350,2.773402e-02
8,LV891,47,3.550,0.4933,0.519436,3.637251e-02
9,LV1699,47,3.550,0.4775,0.523311,1.075235e-02



  Pathways:


,LV,pathway,AUC,FDR
0,LV1058,C2CP_REACTOME_RESPONSE_TO_METAL_IONS,0.998664,5.640270e-02
1,LV1058,C2CP_WP_ZINC_HOMEOSTASIS,0.769963,6.682021e-02
2,LV1189,C2CP_REACTOME_TP53_REGULATES_TRANSCRIPTION_OF_DEATH_RECEPTORS_AND_LIGANDS,0.972506,5.852795e-02
3,LV1282,C2CP_WP_1P36_COPY_NUMBER_VARIATION_SYNDROME,0.836119,6.926451e-06
4,LV1389,C2CP_KEGG_LYSOSOME,0.831486,2.778235e-07
5,LV1389,C2CP_REACTOME_REGULATION_OF_MITF_M_DEPENDENT_GENES_INVOLVED_IN_LYSOSOME_BIOGENESIS_AND_AUTOPHAGY,0.983702,1.822399e-02
6,LV1389,C2CP_WP_DEGRADATION_PATHWAY_OF_SPHINGOLIPIDS_INCLUDING_DISEASES,0.998904,5.568239e-02
7,LV1438,C2CP_REACTOME_CRMPS_IN_SEMA3A_SIGNALING,0.855342,8.864064e-02
8,LV1724,C2CP_KEGG_MEDICUS_REFERENCE_ANTEROGRADE_AXONAL_TRANSPORT,0.804480,8.868812e-02
9,LV194,C2CP_KEGG_P53_SIGNALING_PATHWAY,0.717945,2.972456e-02


In [41]:
pancreas_pathways

,LV,pathway,AUC,FDR
0,LV1058,C2CP_REACTOME_RESPONSE_TO_METAL_IONS,0.998664,5.640270e-02
1,LV1058,C2CP_WP_ZINC_HOMEOSTASIS,0.769963,6.682021e-02
2,LV1189,C2CP_REACTOME_TP53_REGULATES_TRANSCRIPTION_OF_DEATH_RECEPTORS_AND_LIGANDS,0.972506,5.852795e-02
3,LV1282,C2CP_WP_1P36_COPY_NUMBER_VARIATION_SYNDROME,0.836119,6.926451e-06
4,LV1389,C2CP_KEGG_LYSOSOME,0.831486,2.778235e-07
5,LV1389,C2CP_REACTOME_REGULATION_OF_MITF_M_DEPENDENT_GENES_INVOLVED_IN_LYSOSOME_BIOGENESIS_AND_AUTOPHAGY,0.983702,1.822399e-02
6,LV1389,C2CP_WP_DEGRADATION_PATHWAY_OF_SPHINGOLIPIDS_INCLUDING_DISEASES,0.998904,5.568239e-02
7,LV1438,C2CP_REACTOME_CRMPS_IN_SEMA3A_SIGNALING,0.855342,8.864064e-02
8,LV1724,C2CP_KEGG_MEDICUS_REFERENCE_ANTEROGRADE_AXONAL_TRANSPORT,0.804480,8.868812e-02
9,LV194,C2CP_KEGG_P53_SIGNALING_PATHWAY,0.717945,2.972456e-02


In [42]:
pituitary_pathways = analyze("Pituitary", "pituitary")

────────────────────────────────────────────────────────────────
  Pituitary  (keyword: 'pituitary')  |  samples in model: 25
  Expected in top 1%: 0.2  |  Selected LVs: 0


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR



  Pathways:
  (none)


In [43]:
pituitary_pathways

,LV,pathway,AUC,FDR


In [44]:
prostate_pathways = analyze("Prostate", "prostate")

────────────────────────────────────────────────────────────────
  Prostate  (keyword: 'prostate')  |  samples in model: 6,335
  Expected in top 1%: 63.3  |  Selected LVs: 15


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1362,202,3.189,0.4770,0.512236,2.255821e-03
1,LV1561,184,2.905,0.4710,0.527038,2.826329e-12
2,LV299,178,2.810,0.4745,0.525512,4.906036e-11
3,LV1432,174,2.747,0.4762,0.521807,2.423404e-08
4,LV1145,169,2.668,0.4865,0.509832,1.489998e-02
5,LV1172,162,2.557,0.4856,0.520106,2.903725e-07
6,LV1601,161,2.541,0.4775,0.515220,1.206832e-04
7,LV37,157,2.478,0.4854,0.509274,2.240084e-02
8,LV873,154,2.431,0.4856,0.515248,1.177483e-04
9,LV746,149,2.352,0.4788,0.515544,8.576832e-05



  Pathways:


,LV,pathway,AUC,FDR
0,LV1145,C2CP_KEGG_MEDICUS_REFERENCE_MDM2_P21_CELL_CYCLE_G1_S_N00536,0.965898,0.070395
1,LV1151,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_MACHR_CA2_APOPTOTIC_PATHWAY,0.893499,0.061869
2,LV1172,C2CP_REACTOME_MOLECULES_ASSOCIATED_WITH_ELASTIC_FIBRES,1.000000,0.000361
3,LV1172,C2CP_REACTOME_ELASTIC_FIBRE_FORMATION,0.814207,0.016491
4,LV1172,C2CP_WP_MAJOR_RECEPTORS_TARGETED_BY_EPINEPHRINE_AND_NOREPINEPHRINE,0.977844,0.069074
5,LV174,C2CP_REACTOME_ACTIVATED_PKN1_STIMULATES_TRANSCRIPTION_OF_AR_ANDROGEN_RECEPTOR_REGULATED_GENES_KLK2_AND_KLK3,0.697288,0.066117


In [45]:
prostate_pathways

,LV,pathway,AUC,FDR
0,LV1145,C2CP_KEGG_MEDICUS_REFERENCE_MDM2_P21_CELL_CYCLE_G1_S_N00536,0.965898,0.070395
1,LV1151,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_MACHR_CA2_APOPTOTIC_PATHWAY,0.893499,0.061869
2,LV1172,C2CP_REACTOME_MOLECULES_ASSOCIATED_WITH_ELASTIC_FIBRES,1.000000,0.000361
3,LV1172,C2CP_REACTOME_ELASTIC_FIBRE_FORMATION,0.814207,0.016491
4,LV1172,C2CP_WP_MAJOR_RECEPTORS_TARGETED_BY_EPINEPHRINE_AND_NOREPINEPHRINE,0.977844,0.069074
5,LV174,C2CP_REACTOME_ACTIVATED_PKN1_STIMULATES_TRANSCRIPTION_OF_AR_ANDROGEN_RECEPTOR_REGULATED_GENES_KLK2_AND_KLK3,0.697288,0.066117


In [46]:
skin_pathways = analyze("Skin", "skin")

────────────────────────────────────────────────────────────────
  Skin  (keyword: 'skin')  |  samples in model: 10,464
  Expected in top 1%: 104.6  |  Selected LVs: 33


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV135,344,3.288,0.4406,0.541302,5.786093e-46
1,LV937,305,2.915,0.4739,0.521869,7.145982e-14
2,LV1236,295,2.819,0.4943,0.512331,3.067113e-05
3,LV1224,286,2.733,0.4837,0.517112,5.565893e-09
4,LV11,275,2.628,0.4588,0.528980,2.388256e-23
5,LV1138,273,2.609,0.5006,0.508203,5.893561e-03
6,LV1061,273,2.609,0.4677,0.527091,1.382651e-20
7,LV312,264,2.523,0.4760,0.521063,5.780055e-13
8,LV1425,264,2.523,0.5017,0.511274,1.403307e-04
9,LV171,249,2.380,0.4954,0.507519,1.181465e-02



  Pathways:


,LV,pathway,AUC,FDR
0,LV1039,C2CP_BIOCARTA_COMP_PATHWAY,0.968842,0.022100
1,LV1039,C2CP_BIOCARTA_CLASSIC_PATHWAY,0.929976,0.085337
2,LV1061,C2CP_KEGG_GLYCINE_SERINE_AND_THREONINE_METABOLISM,0.889486,0.015817
3,LV1061,C2CP_WP_DISORDERS_OF_FRUCTOSE_METABOLISM,1.000000,0.056403
4,LV11,C2CP_WP_BIOMARKERS_FOR_UREA_CYCLE_DISORDERS,0.998419,0.055862
5,LV11,C2CP_KEGG_MEDICUS_REFERENCE_BILE_ACID_BIOSYNTHESIS,0.987243,0.057616
6,LV11,C2CP_WP_ANDROGEN_BIOSYNTHESIS,0.969225,0.063770
7,LV11,C2CP_REACTOME_SYNTHESIS_OF_BILE_ACIDS_AND_BILE_SALTS_VIA_24_HYDROXYCHOLESTEROL,0.964482,0.066117
8,LV11,C2CP_WP_FARNESOID_X_RECEPTOR_PATHWAY,0.959876,0.068223
9,LV11,C2CP_WP_CHOLESTASIS,0.844082,0.093431


In [47]:
skin_pathways

,LV,pathway,AUC,FDR
0,LV1039,C2CP_BIOCARTA_COMP_PATHWAY,0.968842,0.022100
1,LV1039,C2CP_BIOCARTA_CLASSIC_PATHWAY,0.929976,0.085337
2,LV1061,C2CP_KEGG_GLYCINE_SERINE_AND_THREONINE_METABOLISM,0.889486,0.015817
3,LV1061,C2CP_WP_DISORDERS_OF_FRUCTOSE_METABOLISM,1.000000,0.056403
4,LV11,C2CP_WP_BIOMARKERS_FOR_UREA_CYCLE_DISORDERS,0.998419,0.055862
5,LV11,C2CP_KEGG_MEDICUS_REFERENCE_BILE_ACID_BIOSYNTHESIS,0.987243,0.057616
6,LV11,C2CP_WP_ANDROGEN_BIOSYNTHESIS,0.969225,0.063770
7,LV11,C2CP_REACTOME_SYNTHESIS_OF_BILE_ACIDS_AND_BILE_SALTS_VIA_24_HYDROXYCHOLESTEROL,0.964482,0.066117
8,LV11,C2CP_WP_FARNESOID_X_RECEPTOR_PATHWAY,0.959876,0.068223
9,LV11,C2CP_WP_CHOLESTASIS,0.844082,0.093431


In [48]:
small_intestine_pathways = analyze("Small intestine", "small intestine")

────────────────────────────────────────────────────────────────
  Small intestine  (keyword: 'small intestine')  |  samples in model: 304
  Expected in top 1%: 3.0  |  Selected LVs: 44


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV467,61,20.066,0.3493,0.610196,1.353767e-10
1,LV7,57,18.750,0.4049,0.575432,1.191857e-05
2,LV245,55,18.093,0.3446,0.644541,3.157366e-17
3,LV217,51,16.777,0.2893,0.642512,8.862230e-17
4,LV829,50,16.448,0.3128,0.647851,5.961917e-18
5,LV1017,45,14.803,0.2950,0.635749,2.265524e-15
6,LV342,42,13.816,0.1700,0.711022,1.488095e-34
7,LV909,39,12.829,0.2982,0.648181,5.058000e-18
8,LV1217,38,12.500,0.3876,0.585287,7.036494e-07
9,LV1022,38,12.500,0.3395,0.616316,1.175259e-11



  Pathways:


,LV,pathway,AUC,FDR
0,LV1017,C2CP_KEGG_MEDICUS_ENV_FACTOR_IRON_TO_ANTEROGRADE_AXONAL_TRANSPORT,0.910452,0.049927
1,LV1047,C2CP_KEGG_MEDICUS_REFERENCE_PTH_PTH1R_PKA_SIGNALING_PATHWAY,0.916603,0.097814
2,LV1118,C2CP_WP_HEDGEHOG_SIGNALING_WP4249,0.834746,0.013760
3,LV1118,C2CP_BIOCARTA_COMP_PATHWAY,0.999855,0.016156
4,LV1118,C2CP_REACTOME_ACTIVATION_OF_SMO,1.000000,0.016156
5,LV114,C2CP_REACTOME_KILLING_MECHANISMS,0.933013,0.091309
6,LV1298,C2CP_KEGG_MEDICUS_REFERENCE_IGF2_IGF1R_PI3K_SIGNALING_PATHWAY,1.000000,0.057345
7,LV1298,C2CP_REACTOME_INTERLEUKIN_6_SIGNALING,1.000000,0.057345
8,LV1537,C2CP_REACTOME_CARNITINE_SHUTTLE,0.929781,0.096074
9,LV1543,C2CP_REACTOME_MET_ACTIVATES_PTK2_SIGNALING,0.886230,0.008311


In [49]:
small_intestine_pathways

,LV,pathway,AUC,FDR
0,LV1017,C2CP_KEGG_MEDICUS_ENV_FACTOR_IRON_TO_ANTEROGRADE_AXONAL_TRANSPORT,0.910452,0.049927
1,LV1047,C2CP_KEGG_MEDICUS_REFERENCE_PTH_PTH1R_PKA_SIGNALING_PATHWAY,0.916603,0.097814
2,LV1118,C2CP_WP_HEDGEHOG_SIGNALING_WP4249,0.834746,0.013760
3,LV1118,C2CP_BIOCARTA_COMP_PATHWAY,0.999855,0.016156
4,LV1118,C2CP_REACTOME_ACTIVATION_OF_SMO,1.000000,0.016156
5,LV114,C2CP_REACTOME_KILLING_MECHANISMS,0.933013,0.091309
6,LV1298,C2CP_KEGG_MEDICUS_REFERENCE_IGF2_IGF1R_PI3K_SIGNALING_PATHWAY,1.000000,0.057345
7,LV1298,C2CP_REACTOME_INTERLEUKIN_6_SIGNALING,1.000000,0.057345
8,LV1537,C2CP_REACTOME_CARNITINE_SHUTTLE,0.929781,0.096074
9,LV1543,C2CP_REACTOME_MET_ACTIVATES_PTK2_SIGNALING,0.886230,0.008311


In [50]:
spleen_pathways = analyze("Spleen", "spleen")

────────────────────────────────────────────────────────────────
  Spleen  (keyword: 'spleen')  |  samples in model: 501
  Expected in top 1%: 5.0  |  Selected LVs: 9


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV727,47,9.381,0.4603,0.549886,8.156842e-04
1,LV70,40,7.984,0.3311,0.586178,2.593821e-09
2,LV611,36,7.186,0.4395,0.556788,1.273424e-04
3,LV527,31,6.188,0.4214,0.551378,5.787568e-04
4,LV14,30,5.988,0.4554,0.547540,1.481580e-03
5,LV103,28,5.589,0.4445,0.549160,9.757160e-04
6,LV1635,21,4.192,0.4474,0.536591,1.637668e-02
7,LV573,21,4.192,0.3972,0.559915,4.902470e-05
8,LV1019,20,3.992,0.4474,0.539256,9.791345e-03



  Pathways:


,LV,pathway,AUC,FDR
0,LV1019,C2CP_KEGG_MEDICUS_REFERENCE_PTH_PTH1R_PKA_SIGNALING_PATHWAY,0.957200,0.065724
1,LV103,C2CP_PID_RHODOPSIN_PATHWAY,0.997573,0.055910
2,LV14,C2CP_KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISOME_DISASSEMBLY,0.941648,0.038331
3,LV1635,C2CP_KEGG_MEDICUS_ENV_FACTOR_DCE_TO_DNA_ADDUCTS,0.855302,0.081620
4,LV70,C2CP_WP_STRIATED_MUSCLE_CONTRACTION_PATHWAY,0.902925,0.004298
5,LV727,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_SNCA_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,0.753577,0.039410
6,LV727,C2CP_KEGG_VALINE_LEUCINE_AND_ISOLEUCINE_BIOSYNTHESIS,0.941699,0.081258


In [51]:
spleen_pathways

,LV,pathway,AUC,FDR
0,LV1019,C2CP_KEGG_MEDICUS_REFERENCE_PTH_PTH1R_PKA_SIGNALING_PATHWAY,0.957200,0.065724
1,LV103,C2CP_PID_RHODOPSIN_PATHWAY,0.997573,0.055910
2,LV14,C2CP_KEGG_MEDICUS_REFERENCE_TRAIP_DEPENDENT_REPLISOME_DISASSEMBLY,0.941648,0.038331
3,LV1635,C2CP_KEGG_MEDICUS_ENV_FACTOR_DCE_TO_DNA_ADDUCTS,0.855302,0.081620
4,LV70,C2CP_WP_STRIATED_MUSCLE_CONTRACTION_PATHWAY,0.902925,0.004298
5,LV727,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_SNCA_TO_ELECTRON_TRANSFER_IN_COMPLEX_I,0.753577,0.039410
6,LV727,C2CP_KEGG_VALINE_LEUCINE_AND_ISOLEUCINE_BIOSYNTHESIS,0.941699,0.081258


In [52]:
stomach_pathways = analyze("Stomach", "stomach")

────────────────────────────────────────────────────────────────
  Stomach  (keyword: 'stomach')  |  samples in model: 436
  Expected in top 1%: 4.4  |  Selected LVs: 37


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV125,48,11.009,0.3814,0.575706,2.483966e-07
1,LV290,43,9.863,0.4613,0.547495,1.274104e-03
2,LV1721,42,9.633,0.3715,0.565979,6.836624e-06
3,LV51,39,8.945,0.4085,0.556451,1.241536e-04
4,LV1216,39,8.945,0.3706,0.586919,3.082885e-09
5,LV1453,35,8.028,0.4772,0.546152,1.762662e-03
6,LV1403,34,7.798,0.3991,0.572143,8.707240e-07
7,LV346,33,7.569,0.3428,0.601545,3.857716e-12
8,LV284,32,7.340,0.4825,0.539355,7.831299e-03
9,LV361,31,7.110,0.5103,0.539841,7.110624e-03



  Pathways:


,LV,pathway,AUC,FDR
0,LV1047,C2CP_KEGG_MEDICUS_REFERENCE_PTH_PTH1R_PKA_SIGNALING_PATHWAY,0.916603,0.097814
1,LV1057,C2CP_BIOCARTA_VITCB_PATHWAY,0.934916,0.093600
2,LV12,C2CP_KEGG_MEDICUS_PATHOGEN_SHIGELLA_IPAC_TO_ACTIN_SIGNALING_PATHWAY,0.983742,0.062716
3,LV12,C2CP_KEGG_MEDICUS_REFERENCE_ARNO_ARF_ACTB_G_SIGNALING_PATHWAY,0.797179,0.069078
4,LV1216,C2CP_REACTOME_NEGATIVE_REGULATION_OF_NMDA_RECEPTOR_MEDIATED_NEURONAL_TRANSMISSION,0.937267,0.053183
5,LV125,C2CP_KEGG_MEDICUS_REFERENCE_REGULATION_OF_GF_RTK_RAS_ERK_SIGNALING_PATHWAY_ADAPTOR_PROTEINS,0.933630,0.089650
6,LV1258,C2CP_REACTOME_PHASE_0_RAPID_DEPOLARISATION,1.000000,0.005192
7,LV1258,C2CP_REACTOME_MUSCLE_CONTRACTION,0.637225,0.050150
8,LV1258,C2CP_REACTOME_ION_HOMEOSTASIS,0.726109,0.066564
9,LV1258,C2CP_WP_PANCREATIC_CANCER_SUBTYPES,0.752519,0.092620


In [53]:
stomach_pathways

,LV,pathway,AUC,FDR
0,LV1047,C2CP_KEGG_MEDICUS_REFERENCE_PTH_PTH1R_PKA_SIGNALING_PATHWAY,0.916603,0.097814
1,LV1057,C2CP_BIOCARTA_VITCB_PATHWAY,0.934916,0.093600
2,LV12,C2CP_KEGG_MEDICUS_PATHOGEN_SHIGELLA_IPAC_TO_ACTIN_SIGNALING_PATHWAY,0.983742,0.062716
3,LV12,C2CP_KEGG_MEDICUS_REFERENCE_ARNO_ARF_ACTB_G_SIGNALING_PATHWAY,0.797179,0.069078
4,LV1216,C2CP_REACTOME_NEGATIVE_REGULATION_OF_NMDA_RECEPTOR_MEDIATED_NEURONAL_TRANSMISSION,0.937267,0.053183
5,LV125,C2CP_KEGG_MEDICUS_REFERENCE_REGULATION_OF_GF_RTK_RAS_ERK_SIGNALING_PATHWAY_ADAPTOR_PROTEINS,0.933630,0.089650
6,LV1258,C2CP_REACTOME_PHASE_0_RAPID_DEPOLARISATION,1.000000,0.005192
7,LV1258,C2CP_REACTOME_MUSCLE_CONTRACTION,0.637225,0.050150
8,LV1258,C2CP_REACTOME_ION_HOMEOSTASIS,0.726109,0.066564
9,LV1258,C2CP_WP_PANCREATIC_CANCER_SUBTYPES,0.752519,0.092620


In [54]:
testis_pathways = analyze("Testis", "testis")

────────────────────────────────────────────────────────────────
  Testis  (keyword: 'testis')  |  samples in model: 317
  Expected in top 1%: 3.2  |  Selected LVs: 28


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1466,41,12.934,0.4160,0.574019,3.334786e-05
1,LV969,40,12.619,0.4403,0.569282,1.101555e-04
2,LV990,39,12.303,0.3793,0.584448,1.821272e-06
3,LV793,37,11.672,0.4467,0.559871,9.036201e-04
4,LV1046,37,11.672,0.3452,0.580272,6.367846e-06
5,LV30,34,10.726,0.4165,0.541066,2.794064e-02
6,LV1502,33,10.410,0.4070,0.550928,5.303150e-03
7,LV891,33,10.410,0.4686,0.561584,6.383867e-04
8,LV568,32,10.095,0.4993,0.542487,2.276106e-02
9,LV1507,25,7.887,0.4344,0.557036,1.612297e-03



  Pathways:


,LV,pathway,AUC,FDR
0,LV100,C2CP_KEGG_MEDICUS_REFERENCE_TCR_PLCG_ITPR_SIGNALING_PATHWAY,0.943106,3.503176e-02
1,LV100,C2CP_WP_REGULATION_OF_CYTOTOXIC_T_CELL_RESPONSES_BY_MALAT1_MIR1516_CIRCUIT,0.991172,5.686918e-02
2,LV1028,C2CP_WP_METHIONINE_METABOLISM_LEADING_TO_SULFUR_AMINO_ACIDS_AND_RELATED_DISORDERS,0.963774,7.144923e-02
3,LV1259,C2CP_REACTOME_SYNTHESIS_OF_GLYCOSYLPHOSPHATIDYLINOSITOL_GPI,0.954780,3.542061e-02
4,LV1259,C2CP_BIOCARTA_ATRBRCA_PATHWAY,0.872571,5.031171e-02
5,LV1271,C2CP_PID_INTEGRIN3_PATHWAY,0.786049,2.474244e-02
6,LV1271,C2CP_WP_OSTEOBLAST_SIGNALING,0.998034,5.621459e-02
7,LV1271,C2CP_REACTOME_DISSOLUTION_OF_FIBRIN_CLOT,0.960982,6.852737e-02
8,LV1271,C2CP_BIOCARTA_PLATELETAPP_PATHWAY,0.959262,6.962403e-02
9,LV1483,C2CP_KEGG_MEDICUS_REFERENCE_LOADING_OF_THE_SMC5_SMC6_COMPLEX,0.971932,6.890285e-02


In [55]:
testis_pathways

,LV,pathway,AUC,FDR
0,LV100,C2CP_KEGG_MEDICUS_REFERENCE_TCR_PLCG_ITPR_SIGNALING_PATHWAY,0.943106,3.503176e-02
1,LV100,C2CP_WP_REGULATION_OF_CYTOTOXIC_T_CELL_RESPONSES_BY_MALAT1_MIR1516_CIRCUIT,0.991172,5.686918e-02
2,LV1028,C2CP_WP_METHIONINE_METABOLISM_LEADING_TO_SULFUR_AMINO_ACIDS_AND_RELATED_DISORDERS,0.963774,7.144923e-02
3,LV1259,C2CP_REACTOME_SYNTHESIS_OF_GLYCOSYLPHOSPHATIDYLINOSITOL_GPI,0.954780,3.542061e-02
4,LV1259,C2CP_BIOCARTA_ATRBRCA_PATHWAY,0.872571,5.031171e-02
5,LV1271,C2CP_PID_INTEGRIN3_PATHWAY,0.786049,2.474244e-02
6,LV1271,C2CP_WP_OSTEOBLAST_SIGNALING,0.998034,5.621459e-02
7,LV1271,C2CP_REACTOME_DISSOLUTION_OF_FIBRIN_CLOT,0.960982,6.852737e-02
8,LV1271,C2CP_BIOCARTA_PLATELETAPP_PATHWAY,0.959262,6.962403e-02
9,LV1483,C2CP_KEGG_MEDICUS_REFERENCE_LOADING_OF_THE_SMC5_SMC6_COMPLEX,0.971932,6.890285e-02


In [56]:
thyroid_pathways = analyze("Thyroid", "thyroid")

────────────────────────────────────────────────────────────────
  Thyroid  (keyword: 'thyroid')  |  samples in model: 730
  Expected in top 1%: 7.3  |  Selected LVs: 46


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV1379,98,13.425,0.4684,0.534016,4.548617e-03
1,LV1584,74,10.137,0.4116,0.572724,4.676374e-10
2,LV824,69,9.452,0.4049,0.555848,1.940397e-06
3,LV1198,47,6.439,0.4350,0.538232,1.343598e-03
4,LV122,46,6.302,0.4645,0.528860,1.719018e-02
5,LV669,37,5.069,0.3650,0.588286,2.557595e-14
6,LV910,37,5.069,0.4695,0.534281,4.266444e-03
7,LV626,33,4.521,0.4523,0.561625,1.445480e-07
8,LV300,32,4.384,0.4587,0.534916,3.597841e-03
9,LV993,31,4.247,0.4237,0.549571,2.485281e-05



  Pathways:


,LV,pathway,AUC,FDR
0,LV107,C2CP_BIOCARTA_PCAF_PATHWAY,0.943891,7.657923e-02
1,LV107,C2CP_KEGG_MEDICUS_REFERENCE_N_GLYCAN_BIOSYNTHESIS,0.847796,8.908425e-02
2,LV1198,C2CP_PID_TOLL_ENDOGENOUS_PATHWAY,0.944938,1.308568e-02
3,LV1198,C2CP_REACTOME_NUCLEOTIDE_LIKE_PURINERGIC_RECEPTORS,0.942841,8.164208e-02
4,LV1404,C2CP_WP_T_CELL_MODULATION_AND_DESMOPLASIA_IN_PANCREATIC_CANCER,0.791943,1.194679e-02
5,LV1404,C2CP_WP_EXTRAFOLLICULAR_AND_FOLLICULAR_B_CELL_ACTIVATION_BY_SARSCOV2,0.718821,4.876198e-02
6,LV1404,C2CP_WP_LUPUS_PATHOGENESIS,1.000000,5.904476e-02
7,LV144,C2CP_WP_BMP2WNT4FOXO1_PATHWAY_IN_PRIMARY_ENDOMETRIAL_STROMAL_CELL_DIFFERENTIATION,0.917308,9.781409e-02
8,LV1614,C2CP_REACTOME_DS_GAG_BIOSYNTHESIS,0.997765,5.808118e-02
9,LV1667,C2CP_BIOCARTA_PLATELETAPP_PATHWAY,0.964891,5.895727e-02


In [57]:
thyroid_pathways

,LV,pathway,AUC,FDR
0,LV107,C2CP_BIOCARTA_PCAF_PATHWAY,0.943891,7.657923e-02
1,LV107,C2CP_KEGG_MEDICUS_REFERENCE_N_GLYCAN_BIOSYNTHESIS,0.847796,8.908425e-02
2,LV1198,C2CP_PID_TOLL_ENDOGENOUS_PATHWAY,0.944938,1.308568e-02
3,LV1198,C2CP_REACTOME_NUCLEOTIDE_LIKE_PURINERGIC_RECEPTORS,0.942841,8.164208e-02
4,LV1404,C2CP_WP_T_CELL_MODULATION_AND_DESMOPLASIA_IN_PANCREATIC_CANCER,0.791943,1.194679e-02
5,LV1404,C2CP_WP_EXTRAFOLLICULAR_AND_FOLLICULAR_B_CELL_ACTIVATION_BY_SARSCOV2,0.718821,4.876198e-02
6,LV1404,C2CP_WP_LUPUS_PATHOGENESIS,1.000000,5.904476e-02
7,LV144,C2CP_WP_BMP2WNT4FOXO1_PATHWAY_IN_PRIMARY_ENDOMETRIAL_STROMAL_CELL_DIFFERENTIATION,0.917308,9.781409e-02
8,LV1614,C2CP_REACTOME_DS_GAG_BIOSYNTHESIS,0.997765,5.808118e-02
9,LV1667,C2CP_BIOCARTA_PLATELETAPP_PATHWAY,0.964891,5.895727e-02


In [58]:
uterus_pathways = analyze("Uterus", "uterus")

────────────────────────────────────────────────────────────────
  Uterus  (keyword: 'uterus')  |  samples in model: 254
  Expected in top 1%: 2.5  |  Selected LVs: 19


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV154,77,30.316,0.3366,0.636424,2.515271e-13
1,LV1137,76,29.922,0.2762,0.632052,1.391847e-12
2,LV222,71,27.953,0.1779,0.700933,2.666745e-27
3,LV1591,62,24.410,0.2079,0.666540,3.418694e-19
4,LV378,51,20.079,0.2256,0.694528,1.174200e-25
5,LV348,42,16.536,0.4383,0.590598,1.374396e-06
6,LV58,37,14.567,0.4166,0.587374,3.244134e-06
7,LV347,36,14.174,0.2016,0.646273,3.976953e-15
8,LV587,35,13.780,0.3252,0.654383,1.080231e-16
9,LV492,31,12.205,0.3425,0.600901,7.215632e-08



  Pathways:


,LV,pathway,AUC,FDR
0,LV1177,C2CP_WP_CHOLESTASIS,0.913228,0.054950
1,LV1177,C2CP_WP_DRUG_INDUCTION_OF_BILE_ACID_PATHWAY,1.000000,0.056056
2,LV1180,C2CP_WP_GPCRS_CLASS_B_SECRETINLIKE,0.984452,0.058237
3,LV1180,C2CP_REACTOME_REVERSIBLE_HYDRATION_OF_CARBON_DIOXIDE,0.953410,0.071449
4,LV1180,C2CP_WP_KYNURENINE_PATHWAY_AND_LINKS_TO_CELL_SENESCENCE,0.793972,0.097634
5,LV154,C2CP_REACTOME_LYSINE_CATABOLISM,0.917719,0.086049
6,LV177,C2CP_REACTOME_HEME_BIOSYNTHESIS,0.996484,0.054850
7,LV307,C2CP_WP_TYPE_II_INTERFERON_SIGNALING,0.852473,0.013298
8,LV307,C2CP_BIOCARTA_CLASSIC_PATHWAY,0.983400,0.056869
9,LV492,C2CP_REACTOME_ERYTHROCYTES_TAKE_UP_CARBON_DIOXIDE_AND_RELEASE_OXYGEN,0.998097,0.049667


In [59]:
uterus_pathways

,LV,pathway,AUC,FDR
0,LV1177,C2CP_WP_CHOLESTASIS,0.913228,0.054950
1,LV1177,C2CP_WP_DRUG_INDUCTION_OF_BILE_ACID_PATHWAY,1.000000,0.056056
2,LV1180,C2CP_WP_GPCRS_CLASS_B_SECRETINLIKE,0.984452,0.058237
3,LV1180,C2CP_REACTOME_REVERSIBLE_HYDRATION_OF_CARBON_DIOXIDE,0.953410,0.071449
4,LV1180,C2CP_WP_KYNURENINE_PATHWAY_AND_LINKS_TO_CELL_SENESCENCE,0.793972,0.097634
5,LV154,C2CP_REACTOME_LYSINE_CATABOLISM,0.917719,0.086049
6,LV177,C2CP_REACTOME_HEME_BIOSYNTHESIS,0.996484,0.054850
7,LV307,C2CP_WP_TYPE_II_INTERFERON_SIGNALING,0.852473,0.013298
8,LV307,C2CP_BIOCARTA_CLASSIC_PATHWAY,0.983400,0.056869
9,LV492,C2CP_REACTOME_ERYTHROCYTES_TAKE_UP_CARBON_DIOXIDE_AND_RELEASE_OXYGEN,0.998097,0.049667


In [60]:
vagina_pathways = analyze("Vagina", "vagina")

────────────────────────────────────────────────────────────────
  Vagina  (keyword: 'vagina')  |  samples in model: 167
  Expected in top 1%: 1.7  |  Selected LVs: 9


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV111,33,19.761,0.3608,0.577297,1.353735e-03
1,LV1409,33,19.761,0.4637,0.579587,9.653585e-04
2,LV697,31,18.563,0.2007,0.693621,3.209655e-16
3,LV919,30,17.964,0.2507,0.682524,1.668792e-14
4,LV1608,29,17.366,0.2262,0.657856,4.318099e-11
5,LV302,26,15.569,0.3820,0.627720,8.194049e-08
6,LV228,25,14.970,0.4234,0.599691,3.186639e-05
7,LV1113,22,13.174,0.2073,0.682706,1.663747e-14
8,LV926,21,12.575,0.2462,0.673960,2.481900e-13



  Pathways:


,LV,pathway,AUC,FDR
0,LV1113,C2CP_REACTOME_CD22_MEDIATED_BCR_REGULATION,0.913083,0.000215
1,LV1113,C2CP_REACTOME_CREATION_OF_C4_AND_C2_ACTIVATORS,0.843735,0.001656
2,LV1113,C2CP_REACTOME_ROLE_OF_LAT2_NTAL_LAB_ON_CALCIUM_MOBILIZATION,0.715548,0.056336
3,LV1113,C2CP_REACTOME_SCAVENGING_OF_HEME_FROM_PLASMA,0.717561,0.060492
4,LV1608,C2CP_KEGG_PRIMARY_BILE_ACID_BIOSYNTHESIS,0.997442,0.050150
5,LV302,C2CP_KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.981859,0.000043
6,LV302,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.967561,0.000081
7,LV302,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_SOD1_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.876920,0.000984
8,LV302,C2CP_KEGG_PROTEASOME,0.852715,0.002513
9,LV302,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.876941,0.002513


In [61]:
vagina_pathways

,LV,pathway,AUC,FDR
0,LV1113,C2CP_REACTOME_CD22_MEDIATED_BCR_REGULATION,0.913083,0.000215
1,LV1113,C2CP_REACTOME_CREATION_OF_C4_AND_C2_ACTIVATORS,0.843735,0.001656
2,LV1113,C2CP_REACTOME_ROLE_OF_LAT2_NTAL_LAB_ON_CALCIUM_MOBILIZATION,0.715548,0.056336
3,LV1113,C2CP_REACTOME_SCAVENGING_OF_HEME_FROM_PLASMA,0.717561,0.060492
4,LV1608,C2CP_KEGG_PRIMARY_BILE_ACID_BIOSYNTHESIS,0.997442,0.050150
5,LV302,C2CP_KEGG_MEDICUS_VARIANT_SCRAPIE_CONFORMATION_PRPSC_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.981859,0.000043
6,LV302,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_ABETA_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.967561,0.000081
7,LV302,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_CAUSED_ABERRANT_SOD1_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.876920,0.000984
8,LV302,C2CP_KEGG_PROTEASOME,0.852715,0.002513
9,LV302,C2CP_KEGG_MEDICUS_VARIANT_MUTATION_INACTIVATED_VCP_TO_26S_PROTEASOME_MEDIATED_PROTEIN_DEGRADATION,0.876941,0.002513


In [62]:
whole_blood_pathways = analyze("Whole blood", "whole blood")

────────────────────────────────────────────────────────────────
  Whole blood  (keyword: 'whole blood')  |  samples in model: 13,094
  Expected in top 1%: 130.9  |  Selected LVs: 17


,LV,n_in_top1pct,enrich_ratio,median_rank_pct,AUC,FDR
0,LV147,360,2.749,0.4723,0.528077,4.416171e-27
1,LV127,319,2.436,0.4460,0.529464,1.149726e-29
2,LV1342,317,2.421,0.4875,0.507127,7.305645e-03
3,LV294,301,2.299,0.4667,0.528134,3.555649e-27
4,LV158,300,2.291,0.4883,0.513559,2.508816e-07
5,LV1290,299,2.284,0.4876,0.521207,5.026827e-16
6,LV1375,285,2.177,0.4622,0.530334,2.125829e-31
7,LV1362,276,2.108,0.4946,0.509669,2.596214e-04
8,LV480,276,2.108,0.4518,0.524976,9.213245e-22
9,LV1661,273,2.085,0.4754,0.526394,4.311146e-24



  Pathways:


,LV,pathway,AUC,FDR
0,LV1174,C2CP_KEGG_MEDICUS_REFERENCE_HORMONE_LIKE_CYTOKINE_TO_JAK_STAT_SIGNALING_PATHWAY,0.945302,0.084957
1,LV127,C2CP_BIOCARTA_CLASSIC_PATHWAY,0.955657,0.069624
2,LV1342,C2CP_WP_OVERVIEW_OF_PROINFLAMMATORY_AND_PROFIBROTIC_MEDIATORS,0.737190,0.017584
3,LV1342,C2CP_WP_OVERVIEW_OF_NANOPARTICLE_EFFECTS,0.847604,0.093997
4,LV147,C2CP_REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION,0.673349,0.074990
5,LV1487,C2CP_REACTOME_ATTACHMENT_OF_BACTERIA_TO_EPITHELIAL_CELLS,0.807443,0.082372
6,LV1661,C2CP_REACTOME_TRANSPORT_OF_CONNEXONS_TO_THE_PLASMA_MEMBRANE,0.997554,0.020745
7,LV27,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_LUMINAL_EPITHELIAL_CELLS,0.813456,0.079910
8,LV294,C2CP_REACTOME_SYNTHESIS_OF_PROSTAGLANDINS_PG_AND_THROMBOXANES_TX,0.998142,0.059390
9,LV410,C2CP_REACTOME_SYNTHESIS_OF_BILE_ACIDS_AND_BILE_SALTS_VIA_24_HYDROXYCHOLESTEROL,0.998609,0.054361


In [63]:
whole_blood_pathways

,LV,pathway,AUC,FDR
0,LV1174,C2CP_KEGG_MEDICUS_REFERENCE_HORMONE_LIKE_CYTOKINE_TO_JAK_STAT_SIGNALING_PATHWAY,0.945302,0.084957
1,LV127,C2CP_BIOCARTA_CLASSIC_PATHWAY,0.955657,0.069624
2,LV1342,C2CP_WP_OVERVIEW_OF_PROINFLAMMATORY_AND_PROFIBROTIC_MEDIATORS,0.737190,0.017584
3,LV1342,C2CP_WP_OVERVIEW_OF_NANOPARTICLE_EFFECTS,0.847604,0.093997
4,LV147,C2CP_REACTOME_SIRT1_NEGATIVELY_REGULATES_RRNA_EXPRESSION,0.673349,0.074990
5,LV1487,C2CP_REACTOME_ATTACHMENT_OF_BACTERIA_TO_EPITHELIAL_CELLS,0.807443,0.082372
6,LV1661,C2CP_REACTOME_TRANSPORT_OF_CONNEXONS_TO_THE_PLASMA_MEMBRANE,0.997554,0.020745
7,LV27,C2CP_REACTOME_DEVELOPMENTAL_LINEAGE_OF_MAMMARY_GLAND_LUMINAL_EPITHELIAL_CELLS,0.813456,0.079910
8,LV294,C2CP_REACTOME_SYNTHESIS_OF_PROSTAGLANDINS_PG_AND_THROMBOXANES_TX,0.998142,0.059390
9,LV410,C2CP_REACTOME_SYNTHESIS_OF_BILE_ACIDS_AND_BILE_SALTS_VIA_24_HYDROXYCHOLESTEROL,0.998609,0.054361


In [64]:
_tissue_dfs = {
    "Adipose":                 adipose_pathways,
    "Adrenal gland":           adrenal_gland_pathways,
    "Artery":                  artery_pathways,
    "Bladder":                 bladder_pathways,
    "Blood vessel":            blood_vessel_pathways,
    "Brain":                   brain_pathways,
    "Breast":                  breast_pathways,
    "Cervix":                  cervix_pathways,
    "Colon":                   colon_pathways,
    "Esophagus":               esophagus_pathways,
    "Heart":                   heart_pathways,
    "Kidney":                  kidney_pathways,
    "Liver":                   liver_pathways,
    "Lung":                    lung_pathways,
    "Muscle":                  muscle_pathways,
    "Nerve":                   nerve_pathways,
    "Ovary":                   ovary_pathways,
    "Pancreas":                pancreas_pathways,
    "Pituitary":               pituitary_pathways,
    "Prostate":                prostate_pathways,
    "Skin":                    skin_pathways,
    "Small intestine":         small_intestine_pathways,
    "Spleen":                  spleen_pathways,
    "Stomach":                 stomach_pathways,
    "Testis":                  testis_pathways,
    "Thyroid":                 thyroid_pathways,
    "Uterus":                  uterus_pathways,
    "Vagina":                  vagina_pathways,
    "Whole blood":             whole_blood_pathways,
}

_frames = [df.assign(tissue=t) for t, df in _tissue_dfs.items() if df is not None and not df.empty]
tissues_combined = pd.concat(_frames, ignore_index=True)[["tissue", "LV", "pathway", "AUC", "FDR"]]

_unique_mask = tissues_combined["pathway"].map(
    tissues_combined.groupby("pathway")["tissue"].nunique() == 1
)
tissues_unique = (
    tissues_combined[_unique_mask]
    .sort_values(["tissue", "FDR"])
    .reset_index(drop=True)
)

display(tissues_unique)

,tissue,LV,pathway,AUC,FDR
0,Adipose,LV262,C2CP_REACTOME_RESPONSE_OF_EIF2AK4_GCN2_TO_AMINO_ACID_DEFICIENCY,0.946340,5.910003e-11
1,Adipose,LV262,C2CP_REACTOME_SARS_COV_2_MODULATES_HOST_TRANSLATION_MACHINERY,0.921941,4.695374e-05
2,Adipose,LV262,C2CP_REACTOME_SARS_COV_1_MODULATES_HOST_TRANSLATION_MACHINERY,0.889413,3.134005e-03
3,Adipose,LV256,C2CP_REACTOME_DEVELOPMENTAL_LINEAGES_OF_THE_MAMMARY_GLAND,0.929677,3.142651e-03
4,Adipose,LV758,C2CP_WP_PROSTAGLANDIN_SYNTHESIS_AND_REGULATION,0.901514,3.169349e-03
5,Adipose,LV1286,C2CP_BIOCARTA_P53_PATHWAY,0.994411,1.676426e-02
6,Adipose,LV1599,C2CP_REACTOME_RHO_GTPASES_ACTIVATE_IQGAPS,0.894907,1.839603e-02
7,Adipose,LV73,C2CP_WP_LTF_DANGER_SIGNAL_RESPONSE_PATHWAY,0.982757,2.001041e-02
8,Adipose,LV1659,C2CP_REACTOME_METABOLISM_OF_PORPHYRINS,0.913650,2.315255e-02
9,Adipose,LV272,C2CP_KEGG_MEDICUS_REFERENCE_KEAP1_NRF2_SIGNALING_PATHWAY,0.895403,2.779711e-02
